# SLAP2 oddball analysis: mouse 803496, DMD1

Session `sub-803496_ses-SLAP2-803496-2025-07-02-15-33-26` from DANDI dandiset 001424.

## 1. Setup

Imports and the session to analyse.

In [ ]:
# First-time setup (run once in a terminal):
#   pip install pynwb tqdm requests
#   pip install --upgrade dandi

from pathlib import Path
import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import requests
from tqdm import tqdm
from dandi import dandiapi
from pynwb import NWBHDF5IO

# --- Where the data file is stored ---
# (before publishing on GitHub, change this to a relative folder such as Path('data'))
output_dir = Path('/Users/aletheahuang/content')
output_dir.mkdir(exist_ok=True)

# --- Session to analyse ---
dandiset_id = "001424"
dandi_filepath = "sub-803496/sub-803496_ses-SLAP2-803496-2025-07-02-15-33-26_image+ophys.nwb"
filename = dandi_filepath.split("/")[-1]
filepath = output_dir / filename

## 2. Download and open the NWB file

In [ ]:
# --- Download the NWB file from DANDI if it is not already on disk ---
if filepath.exists():
    print(f"{filename} already exists.")
else:
    client = dandiapi.DandiAPIClient()
    dandiset = client.get_dandiset(dandiset_id)
    file = dandiset.get_asset_by_path(dandi_filepath)
    file_url = file.download_url

    # Stream download
    response = requests.get(file_url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024  # 1 Kibibyte

    with open(filepath, 'wb') as f, tqdm(
        desc=f"Downloading {filename}",
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(block_size):
            f.write(data)
            bar.update(len(data))

    print(f"Downloaded file to {filepath}")

# --- Open the NWB file ---
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=UserWarning,
        message=".*Date is missing timezone information.*",
    )
    io = NWBHDF5IO(filepath, mode="r", load_namespaces=True)
    nwb = io.read()

## 3. Session metadata and file contents

In [ ]:
# --- Session and subject ---
print("Session :", nwb.session_id)
print("Subject :", nwb.subject.subject_id, "|", nwb.subject.genotype, "|", nwb.subject.sex, "|", nwb.subject.age)
print()

# --- What is stored in the file ---
print("PROCESSING MODULES")
for module_name, module in nwb.processing.items():
    print(f"\nProcessing module: {module_name}  (type: {type(module).__name__})")
    
    di_keys = list(module.data_interfaces.keys())
    print(di_keys)
    
    for di_name, di in module.data_interfaces.items():
        print(f"    - Interface: {di_name}  (type: {type(di).__name__})")

        if hasattr(di, "roi_response_series"):
            print("        ROI response series:", list(di.roi_response_series.keys()))

# Handle to the ophys module (dF/F traces and ROI masks live here)
ophys = nwb.processing['ophys']

## 4. ROI segmentation (DMD1 and DMD2)

In [ ]:
# -----------------------------
# Robust getters / converters
# -----------------------------
def get_mask_column(plane_seg, colname: str):
    """Fetch PlaneSegmentation column whether stored as table column or attribute."""
    if hasattr(plane_seg, "colnames") and (colname in list(plane_seg.colnames)):
        try:
            return plane_seg[colname]
        except KeyError:
            pass
    return getattr(plane_seg, colname, None)

def pixelmask_to_array(pm):
    """
    Convert one ROI's pixel_mask into a numeric array of shape (K,3): [y, x, weight].
    """
    if pm is None:
        return np.zeros((0, 3), dtype=float)

    arr = np.asarray(pm)

    # structured dtype
    if getattr(arr.dtype, "names", None) is not None:
        names = list(arr.dtype.names)

        if "y" in names and "x" in names:
            y = arr["y"].astype(float)
            x = arr["x"].astype(float)
        else:
            if names[0].lower() == "x" and len(names) > 1 and names[1].lower() == "y":
                x = arr[names[0]].astype(float)
                y = arr[names[1]].astype(float)
            else:
                y = arr[names[0]].astype(float)
                x = arr[names[1]].astype(float)

        if "weight" in names:
            w = arr["weight"].astype(float)
        elif len(names) >= 3:
            w = arr[names[2]].astype(float)
        else:
            w = np.ones_like(y, dtype=float)

        return np.column_stack([y, x, w])

    # numeric ndarray
    if arr.ndim == 2 and arr.shape[1] >= 2:
        if arr.shape[1] == 2:
            w = np.ones((arr.shape[0], 1), dtype=float)
            return np.concatenate([arr.astype(float), w], axis=1)
        return arr[:, :3].astype(float)

    # list-like
    try:
        lst = list(pm)
        if len(lst) == 0:
            return np.zeros((0, 3), dtype=float)
        arr2 = np.array(lst, dtype=float)
        if arr2.ndim == 1:
            arr2 = arr2.reshape(1, -1)
        if arr2.shape[1] == 2:
            w = np.ones((arr2.shape[0], 1), dtype=float)
            arr2 = np.concatenate([arr2, w], axis=1)
        return arr2[:, :3].astype(float)
    except Exception:
        return np.zeros((0, 3), dtype=float)

# -----------------------------
# Build label images
# -----------------------------
def label_from_image_mask(img_col, n_rois: int):
    first = np.asarray(img_col[0])
    H, W = first.shape
    label = np.zeros((H, W), dtype=np.int32)

    for i in range(n_rois):
        m = np.asarray(img_col[i])

        fg = (m > 0)
        if fg.sum() == 0:
            fg = np.isfinite(m) & (m != 0)
        if fg.sum() == 0:
            fg = np.isfinite(m)

        label[fg] = i + 1

    return label

def label_from_pixel_mask(pix_col, n_rois: int):
    max_y, max_x = 0, 0
    cached = []
    for i in range(n_rois):
        pm = pixelmask_to_array(pix_col[i])
        cached.append(pm)
        if pm.shape[0] == 0:
            continue
        max_y = max(max_y, int(np.max(pm[:, 0])))
        max_x = max(max_x, int(np.max(pm[:, 1])))

    label = np.zeros((max_y + 1, max_x + 1), dtype=np.int32)

    for i, pm in enumerate(cached):
        if pm.shape[0] == 0:
            continue
        ys = pm[:, 0].astype(int)
        xs = pm[:, 1].astype(int)
        label[ys, xs] = i + 1

    return label

def make_label_image(plane_seg):
    roi_ids = np.asarray(plane_seg.id[:])
    n = len(roi_ids)

    img_col = get_mask_column(plane_seg, "image_mask")
    if img_col is not None:
        _ = np.asarray(img_col[0]).shape
        return label_from_image_mask(img_col, n), roi_ids

    pix_col = get_mask_column(plane_seg, "pixel_mask")
    if pix_col is not None:
        return label_from_pixel_mask(pix_col, n), roi_ids

    raise RuntimeError(
        f"No readable image_mask or pixel_mask. Available columns: {list(getattr(plane_seg, 'colnames', []))}"
    )

# -----------------------------
# Display helpers
# -----------------------------
def apply_display_transform(label_img, rotate_k=0, flip_ud=False, flip_lr=False):
    out = label_img
    if rotate_k:
        out = np.rot90(out, k=rotate_k)
    if flip_ud:
        out = np.flipud(out)
    if flip_lr:
        out = np.fliplr(out)
    return out

def crop_to_nonzero(label_img, pad=20):
    """
    Crop to the bounding box of all nonzero ROI pixels, with padding.
    """
    ys, xs = np.where(label_img > 0)
    if len(xs) == 0 or len(ys) == 0:
        return label_img

    y0 = max(0, ys.min() - pad)
    y1 = min(label_img.shape[0], ys.max() + pad + 1)
    x0 = max(0, xs.min() - pad)
    x1 = min(label_img.shape[1], xs.max() + pad + 1)

    return label_img[y0:y1, x0:x1]

def center_on_square_canvas(label_img, extra_pad=10):
    """
    Put cropped ROI image onto a square canvas so it appears centered.
    """
    h, w = label_img.shape
    side = max(h, w) + 2 * extra_pad
    canvas = np.zeros((side, side), dtype=label_img.dtype)

    y0 = (side - h) // 2
    x0 = (side - w) // 2
    canvas[y0:y0+h, x0:x0+w] = label_img
    return canvas

def upscale_label_image(label_img, scale=2):
    """
    Enlarge ROI pixels for display only.
    Keeps labels intact by repeating pixels.
    """
    if scale <= 1:
        return label_img
    return np.kron(label_img, np.ones((scale, scale), dtype=label_img.dtype))

def prepare_for_display(label_img, rotate_k=0, flip_ud=False, flip_lr=False,
                        crop_pad=20, center=True, upscale=3):
    out = apply_display_transform(label_img, rotate_k=rotate_k, flip_ud=flip_ud, flip_lr=flip_lr)
    out = crop_to_nonzero(out, pad=crop_pad)
    if center:
        out = center_on_square_canvas(out, extra_pad=10)
    out = upscale_label_image(out, scale=upscale)
    return out

# -----------------------------
# Main display
# -----------------------------
def show_segmentation_dmd1_dmd2(
    nwb_path,
    dmd1_rotate_k=1,
    dmd2_rotate_k=0,
    dmd1_flip_ud=False,
    dmd1_flip_lr=False,
    dmd2_flip_ud=False,
    dmd2_flip_lr=False,
    crop_pad=20,
    upscale=4,
    figsize=(13, 6)
):
    io = NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True)
    try:
        nwb = io.read()
        img_seg = nwb.processing["ophys"].data_interfaces["ImageSegmentation"]
        plane_segs = img_seg.plane_segmentations

        keys = list(plane_segs.keys())
        print("Plane segmentations found:", keys)

        dmd1_key = next((k for k in keys if "DMD1" in k), None)
        dmd2_key = next((k for k in keys if "DMD2" in k), None)
        if dmd1_key is None or dmd2_key is None:
            raise RuntimeError(f"Could not find both DMD1 and DMD2 keys in: {keys}")

        label1, roi_ids1 = make_label_image(plane_segs[dmd1_key])
        label2, roi_ids2 = make_label_image(plane_segs[dmd2_key])

        label1_disp = prepare_for_display(
            label1,
            rotate_k=dmd1_rotate_k,
            flip_ud=dmd1_flip_ud,
            flip_lr=dmd1_flip_lr,
            crop_pad=crop_pad,
            center=True,
            upscale=upscale
        )

        label2_disp = prepare_for_display(
            label2,
            rotate_k=dmd2_rotate_k,
            flip_ud=dmd2_flip_ud,
            flip_lr=dmd2_flip_lr,
            crop_pad=crop_pad,
            center=True,
            upscale=upscale
        )

        fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

        im1 = axes[0].imshow(label1_disp, cmap="viridis", interpolation="nearest")
        axes[0].set_title(f"{dmd1_key} (N={len(roi_ids1)})")
        axes[0].axis("off")
        cbar1 = plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
        cbar1.set_label("ROI Index")

        im2 = axes[1].imshow(label2_disp, cmap="viridis", interpolation="nearest")
        axes[1].set_title(f"{dmd2_key} (N={len(roi_ids2)})")
        axes[1].axis("off")
        cbar2 = plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
        cbar2.set_label("ROI Index")
        
        fig.savefig("dmd_segmentation.svg", format="svg", bbox_inches="tight")

        plt.show()
        return fig

    finally:
        io.close()


show_segmentation_dmd1_dmd2(filepath, crop_pad=15, upscale=7)

## 5. Stimulus table and oddball block detection

Finds the two oddball blocks as long runs with at most three orientations and one dominant orientation, then labels each trial within its block.

In [ ]:
# --- Stimulus table ---
stim_table_name = "gratings"
stim_table = nwb.intervals[stim_table_name].to_dataframe()
display(stim_table[:10])

# --- Step 1: orientation in degrees ---
stim_table['orientation_deg'] = (np.degrees(stim_table['orientation']) % 180).round().astype(int)
ori = stim_table['orientation_deg'].values

# --- Parameters (tweakable) ---
max_unique   = 3
min_dom_frac = 0.8
min_block_len = 30
min_gap_after_block = 10   # enforce a small gap before searching for the 2nd block

def find_low_entropy_block(ori, start_idx=0, max_unique=3, min_dom_frac=0.8, min_block_len=30):
    """Find the longest window starting at/after start_idx with ≤ max_unique orientations
    and with a dominant orientation fraction ≥ min_dom_frac once length ≥ min_block_len.
    Returns (start, end) or (None, None)."""
    best_start = None
    best_end = None
    i = start_idx
    n = len(ori)
    while i < n:
        counts = Counter()
        unique_ors = set()
        j = i
        while j < n:
            val = ori[j]
            counts[val] += 1
            unique_ors = set(counts.keys())
            if len(unique_ors) > max_unique:
                break
            total = j - i + 1
            dom_val, dom_count = counts.most_common(1)[0]
            if total >= min_block_len and (dom_count / total) >= min_dom_frac:
                if best_end is None or (j - i) > (best_end - best_start):
                    best_start, best_end = i, j
            j += 1
        i += 1
    return best_start, best_end

# --- Find first oddball block ---
b1_start, b1_end = find_low_entropy_block(
    ori, 0, max_unique=max_unique, min_dom_frac=min_dom_frac, min_block_len=min_block_len
)
if b1_start is None:
    raise RuntimeError("No valid low-entropy block found (block 1)")

# --- Find second oddball block (starts after a gap following block 1) ---
search_start_2 = b1_end + 1 + min_gap_after_block
b2_start, b2_end = (None, None)
if search_start_2 < len(ori):
    b2_start, b2_end = find_low_entropy_block(
        ori, search_start_2, max_unique=max_unique, min_dom_frac=min_dom_frac, min_block_len=min_block_len
    )

if b2_start is None:
    raise RuntimeError("Second low-entropy block not found")

# --- Extract both blocks and classify within each using that block's dominant orientation ---
def classify_block(stim_df):
    dominant_ori = stim_df['orientation_deg'].mode().iloc[0]
    def classify_stim(row):
        if row['contrast'] == 0.0 and row['temporal_frequency'] == 0.0:
            return 'blank'
        elif row['temporal_frequency'] == 0.0:
            return 'static'
        elif row['orientation_deg'] == dominant_ori:
            return 'standard'
        else:
            return str(row['orientation_deg'])
    out = stim_df.copy()
    out['dominant_ori'] = dominant_ori
    out['stim_type'] = out.apply(classify_stim, axis=1)
    return out, dominant_ori

block1 = stim_table.iloc[np.arange(b1_start, b1_end + 1)].copy()
block2 = stim_table.iloc[np.arange(b2_start, b2_end + 1)].copy()

block1_labeled, dom1 = classify_block(block1)
block1_labeled['block_id'] = 1

block2_labeled, dom2 = classify_block(block2)
block2_labeled['block_id'] = 2

# ---
oddball_blocks = pd.concat([block1_labeled, block2_labeled], axis=0, ignore_index=True)

# --- Summaries ---
print(f"Block 1: trials={len(block1_labeled)}, idx=[{b1_start},{b1_end}], dominant={dom1}°")
print(block1_labeled['stim_type'].value_counts())
print()
print(f"Block 2: trials={len(block2_labeled)}, idx=[{b2_start},{b2_end}], dominant={dom2}°")
print(block2_labeled['stim_type'].value_counts())

## 6. Block figure (Fig. 1c)

In [ ]:
# ---- keep text editable in Adobe Illustrator ----
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

def plot_detected_oddball_blocks_publication(
    stim_table,
    block1_labeled,
    block2_labeled,
    save_path=None,
    show_title=False
):
    # --- Global style knobs (paper-friendly) ---
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 14,
        "axes.titlesize": 18,
        "axes.labelsize": 17,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "legend.fontsize": 12,
        "axes.linewidth": 1.2,
    })

    # colors
    c_all = "0.75"
    c_b1 = "#D62728"   # deep red
    c_b2 = "#1F77B4"   # deep blue

    fig, ax = plt.subplots(figsize=(14, 4.8), constrained_layout=True)

    # --- Plot all trials ---
    ax.plot(
        stim_table["start_time"],
        stim_table["orientation_deg"],
        color=c_all,
        linewidth=1.0,
        alpha=0.7,
        label="All trials",
        zorder=1
    )

    # --- Block ranges ---
    b1_start_t = block1_labeled["start_time"].iloc[0]
    b1_end_t   = block1_labeled["start_time"].iloc[-1]
    b2_start_t = block2_labeled["start_time"].iloc[0]
    b2_end_t   = block2_labeled["start_time"].iloc[-1]

    # --- light shaded spans ---
    ax.axvspan(b1_start_t, b1_end_t, color=c_b1, alpha=0.08, lw=0, zorder=0)
    ax.axvspan(b2_start_t, b2_end_t, color=c_b2, alpha=0.08, lw=0, zorder=0)

    # --- Plot block traces ---
    ax.plot(
        block1_labeled["start_time"],
        block1_labeled["orientation_deg"],
        color=c_b1,
        linewidth=2.0,
        label="Oddball block 1",
        zorder=3
    )
    ax.plot(
        block2_labeled["start_time"],
        block2_labeled["orientation_deg"],
        color=c_b2,
        linewidth=2.0,
        label="Oddball block 2",
        zorder=3
    )

    # --- Block boundaries: only one legend entry ---
    ax.axvline(b1_start_t, color=c_b1, linestyle="--", linewidth=1.4, alpha=0.9, zorder=2)
    ax.axvline(b1_end_t,   color=c_b1, linestyle="--", linewidth=1.4, alpha=0.9, zorder=2)

    ax.axvline(b2_start_t, color=c_b2, linestyle="--", linewidth=1.4, alpha=0.9, zorder=2, label="Block boundaries")
    ax.axvline(b2_end_t,   color=c_b2, linestyle="--", linewidth=1.4, alpha=0.9, zorder=2)

    # --- Labels ---
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Orientation (deg)")

    if show_title:
        ax.set_title("Detected oddball blocks")

    # --- Axis range / ticks ---
    ax.set_ylim(-5, 165)
    ax.set_yticks([0, 45, 90, 135, 160])

    xmax = stim_table["start_time"].max()
    ax.set_xticks(np.arange(0, xmax + 1, 500))

    # --- Remove grid ---
    ax.grid(False)

    # --- cleaner spines ---
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # --- legend outside ---
    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=False,
        handlelength=2.8
    )

    # --- save for paper ---
    if save_path is not None:
        fig.savefig(f"{save_path}.pdf", bbox_inches="tight")
        fig.savefig(f"{save_path}.png", dpi=600, bbox_inches="tight")
        fig.savefig(f"{save_path}.svg", dpi=600, bbox_inches="tight")

    plt.show()

plot_detected_oddball_blocks_publication(
    stim_table,
    block1_labeled,
    block2_labeled,
    save_path="figure1a_detected_oddball_blocks",
    show_title=False
)

## 7. Block time ranges and block stimulus tables

In [ ]:
# --- Block time ranges (first and last stimulus onset of each block) ---
block1_t_start = float(block1_labeled['start_time'].iloc[0])
block1_t_end   = float(block1_labeled['start_time'].iloc[-1])

print(f"Block 1: trial idx = [{b1_start}, {b1_end}]  (n={len(block1_labeled)})")
print(f"Block 1 time range: {block1_t_start:.3f} s → {block1_t_end:.3f} s")

block2_t_start = float(block2_labeled['start_time'].iloc[0])
block2_t_end   = float(block2_labeled['start_time'].iloc[-1])

print(f"Block 2: trial idx = [{b2_start}, {b2_end}]  (n={len(block2_labeled)})")
print(f"Block 2 time range: {block2_t_start:.3f} s → {block2_t_end:.3f} s")

# --- Stimulus table for each block ---
block1_stim_table = stim_table[
    (stim_table['start_time'] >= block1_t_start) &
    (stim_table['start_time'] <= block1_t_end)
].copy()

block2_stim_table = stim_table[
    (stim_table['start_time'] >= block2_t_start) &
    (stim_table['start_time'] <= block2_t_end)
].copy()

# --- Keep only the columns the analysis needs ---
# From here on, everything is extracted from block1_stim_table2 and block2_stim_table2
coloumns_need = ['start_time', 'stop_time', 'stim_id', 'orientation_deg','temporal_frequency', 'contrast']
block1_stim_table2 = block1_stim_table[coloumns_need].copy()
block2_stim_table2 = block2_stim_table[coloumns_need].copy()

display(block1_stim_table2.head())
display(block2_stim_table2.head())

## 8. Check of the block edges (read-only)

In [ ]:
# Read-only check of the block edges. Nothing here changes any variable used later.
# It shows the trials just outside and just inside each block, so you can see whether
# the detector picked up a neighbouring trial that belongs to another block.

def show_block_edges(stim_table, start, end, n=3):
    cols = [k for k in ['start_time', 'stim_id', 'diameter', 'contrast',
                        'orientation_deg', 'slap2_trial_idx'] if k in stim_table.columns]
    display(stim_table.iloc[max(0, start - n): start + n][cols])        # around the first trial
    display(stim_table.iloc[max(0, end - n + 1): end + n + 1][cols])    # around the last trial

print(f"Block 1 edges (first trial index {b1_start}, last trial index {b1_end})")
show_block_edges(stim_table, b1_start, b1_end)
print(f"Block 2 edges (first trial index {b2_start}, last trial index {b2_end})")
show_block_edges(stim_table, b2_start, b2_end)

if 'diameter' in stim_table.columns:
    print('Trials whose diameter differs from the rest of their block:')
    print('  Block 1:', int((block1_labeled['diameter'] != block1_labeled['diameter'].mode().iloc[0]).sum()))
    print('  Block 2:', int((block2_labeled['diameter'] != block2_labeled['diameter'].mode().iloc[0]).sum()))

## Overview of the analysis

| Section | Question |
|---|---|
| 9–11 | Load the traces and build the trial-level table that every analysis reads from |
| 12 | What do single trials look like? |
| 13 | What does the trial-averaged response look like, and how much of the difference between conditions is chance? |
| 14 | Is each ROI driven by the grating, and does repetition or surprise change the response? (control vs standard vs oddball, role switch, tuning) |
| 15 | How do single-trial responses vary? |
| 16 | **Standard vs oddball:** per-ROI scatter and fits, slope checks, normalisation, pooled test |
| 17 | **Around the oddball:** the standard before, the mismatch trial, the standard after |
| 18–19 | **Baseline:** the dissertation analysis, then four checks of whether it is more than a built-in effect |
| 20 | **Across blocks:** baseline, early vs late, before vs after the block switch |
| 21 | **Adaptation** over repetitions of the standard |
| 22 | Summary table and export for the across-mouse analysis |

**Chance ranges.** Mismatch conditions have about 30 trials and the standard about 1,400. Wherever a grey band or a "chance" column appears, it shows what the same number of randomly drawn standard trials gives.

## 9. Load dF/F and cut out the two oddball blocks

Loads the DMD1 dF/F traces, prints basic quality numbers, and cuts the traces to each block's time range.

In [ ]:
ophys = nwb.processing['ophys']

dmd1_dff_series = ophys['Fluorescence_DMD1'].roi_response_series['DMD1_dFF']
dmd1_f0_series  = ophys['Fluorescence_DMD1'].roi_response_series['DMD1_F0']

dmd1_dff = np.asarray(dmd1_dff_series.data, float)
t_dmd1   = np.asarray(dmd1_dff_series.timestamps, float)
if dmd1_dff.ndim == 1:
    dmd1_dff = dmd1_dff[:, None]
t1 = t_dmd1

# --- basic quality numbers ---
T, N = dmd1_dff.shape
duration = float(t1[-1] - t1[0])
fs_est = (T - 1) / duration if duration > 0 else np.nan
print(f"[DMD1] dF/F shape: {dmd1_dff.shape}  ({N} ROIs)")
print(f"[DMD1] duration = {duration:.1f} s, sampling rate ≈ {fs_est:.1f} Hz")
print(f"[DMD1] finite fraction = {np.isfinite(dmd1_dff).mean():.2%}")
print("[DMD1] NaNs per ROI:", np.isnan(dmd1_dff).sum(axis=0).tolist())
print(f"[DMD1] global mean = {np.nanmean(dmd1_dff):.4g}, global std = {np.nanstd(dmd1_dff):.4g}")

# --- cut the traces to each block (same time ranges as before) ---
b1_window = (t1 >= block1_t_start) & (t1 <= block1_t_end)
dff1_b1, t1_b1 = dmd1_dff[b1_window, :], t1[b1_window]

b2_window = (t1 >= block2_t_start) & (t1 <= block2_t_end)
dff1_b2, t1_b2 = dmd1_dff[b2_window, :], t1[b2_window]

print(f"\nBlock 1 dF/F: {dff1_b1.shape}, time {t1_b1[0]:.3f} → {t1_b1[-1]:.3f} s")
print(f"Block 2 dF/F: {dff1_b2.shape}, time {t1_b2[0]:.3f} → {t1_b2[-1]:.3f} s")

## 10. Analysis settings and shared helpers

One place for the window lengths, colours, figure folder and small helper functions that every later section uses.

In [ ]:
# ---------------- settings ----------------
PRE_WIN  = 0.343      # s, pre-stimulus baseline window
POST_WIN = 0.343      # s, window after stimulus offset
MIN_POINTS       = 3      # minimum samples in a window
MAX_NAN_FRAC     = 0.5    # relative/baseline rule: drop trial if more than this fraction is NaN
MIN_FINITE_FRAC  = 0.5    # absolute rule: keep trial if at least this fraction is finite

N_DRAWS     = 1000    # random draws for every chance distribution
RANDOM_SEED = 0

MOUSE_ID = str(nwb.subject.subject_id)
DMD      = "DMD1"
N_ROI    = dmd1_dff.shape[1]
EXAMPLE_ROI = 0

# False reproduces the dissertation numbers exactly.
# True removes trials the block detector picked up from a neighbouring block (trial 959 here).
USE_ONLY_IN_BLOCK_TRIALS = False

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

BLOCKS = {
    1: dict(dff=dff1_b1, t=t1_b1, stim=block1_stim_table2, labeled=block1_labeled, standard_ori=int(dom1)),
    2: dict(dff=dff1_b2, t=t1_b2, stim=block2_stim_table2, labeled=block2_labeled, standard_ori=int(dom2)),
}

try:
    from scipy import stats as sp_stats
except ImportError:
    sp_stats = None
    print("scipy is not installed: p-values from t-tests will be left empty.")


# ---------------- small helpers ----------------
def save_fig(fig, name):
    """Save a figure as SVG in the figures folder."""
    fig.savefig(FIG_DIR / f"{name}.svg", format="svg", bbox_inches="tight")


def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)


def oddball_conditions(block):
    """Oddball conditions of a block, the one furthest from the standard first (90° in B1, 0° in B2)."""
    std = BLOCKS[block]["standard_ori"]
    stim = BLOCKS[block]["stim"]
    oris = sorted(set(stim.loc[stim["contrast"] != 0.0, "orientation_deg"]) - {std})
    def distance(o):
        d = abs(int(o) - std) % 180
        return min(d, 180 - d)
    oris = sorted(oris, key=distance, reverse=True)
    return [f"oddball_{int(o)}" for o in oris]


def mismatch_conditions(block):
    """The non-standard conditions of a block: the two oddballs, then omission."""
    return oddball_conditions(block) + ["omission"]


def cond_label(block, condition):
    if condition == "standard":
        return f"Standard {BLOCKS[block]['standard_ori']}°"
    if condition == "omission":
        return "Omission"
    return f"Oddball {condition.split('_')[1]}°"


def cond_color(block, condition):
    if condition == "standard":
        return "black"
    if condition == "omission":
        return "orange"
    return "blue" if condition == oddball_conditions(block)[0] else "red"


def linfit(x, y):
    """Least-squares line y = a*x + b over finite pairs. Returns slope, intercept, Pearson r, n."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2:
        return np.nan, np.nan, np.nan, int(m.sum())
    a, b = np.polyfit(x[m], y[m], 1)
    r = float(np.corrcoef(x[m], y[m])[0, 1])
    return float(a), float(b), r, int(m.sum())


def pearson(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    return float(np.corrcoef(x[m], y[m])[0, 1])


def sem(a, axis=None):
    a = np.asarray(a, float)
    n = np.sum(np.isfinite(a), axis=axis)
    return np.nanstd(a, axis=axis, ddof=1) / np.sqrt(n)


def nanmean_quiet(a, axis=None):
    """np.nanmean without the warning for all-NaN slices."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(a, axis=axis)


def chance_p(value, dist):
    """Two-sided p-value of a real value against a chance distribution:
    how often chance is at least as far from its own typical value."""
    dist = np.asarray(dist, float)
    dist = dist[np.isfinite(dist)]
    if dist.size == 0 or not np.isfinite(value):
        return np.nan
    centre = np.median(dist)
    return float((np.sum(np.abs(dist - centre) >= abs(value - centre)) + 1) / (dist.size + 1))


def paired_p(a, b=0.0):
    """Paired t-test across ROIs (two-sided). ROIs of one neuron are not fully independent,
    so read these p-values as a guide, not as proof."""
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[np.isfinite(d)]
    if d.size < 3 or sp_stats is None:
        return np.nan
    return float(sp_stats.ttest_1samp(d, 0.0).pvalue)


for _b in BLOCKS:
    print(f"Block {_b}: standard = {BLOCKS[_b]['standard_ori']}°, mismatch conditions = {mismatch_conditions(_b)}")

## 11. Trial-level table

One row per trial per ROI. Every later analysis reads from this table.

- `base`: mean dF/F in the pre-stimulus window `[start − 0.343 s, start)`
- `resp`: mean dF/F in the stimulus window `[start, stop)` (the **absolute response**)
- `rel`: `resp − base` (the **relative response**)
- `post`: mean dF/F in `[stop, stop + 0.343 s)`
- `valid_rel`: trial usable for relative and baseline analyses (both windows are good)
- `valid_abs`: trial usable for absolute analyses (the stimulus window is good)

The absolute and relative analyses therefore use slightly different sets of trials.

In [ ]:
def build_trial_table(dff, t, stim_tbl, block_id, standard_ori, not_in_block_ids=(),
                      pre_win=PRE_WIN, post_win=POST_WIN,
                      min_points=MIN_POINTS, max_nan_frac=MAX_NAN_FRAC, min_finite_frac=MIN_FINITE_FRAC):
    t = np.asarray(t, float)
    if not np.all(np.diff(t) >= 0):
        raise ValueError("Timestamps must be sorted in time.")

    tbl = stim_tbl.sort_values("start_time")
    trial_id = tbl.index.to_numpy()
    starts   = tbl["start_time"].to_numpy(float)
    stops    = tbl["stop_time"].to_numpy(float)
    ori      = tbl["orientation_deg"].to_numpy()
    contrast = tbl["contrast"].to_numpy(float)
    n_trials, n_roi = len(tbl), dff.shape[1]

    # ---- trial labels ----
    role = np.where(contrast == 0.0, "omission",
                    np.where(ori == standard_ori, "standard", "oddball"))
    condition = np.array([f"oddball_{int(o)}" if r == "oddball" else r for r, o in zip(role, ori)])

    # ---- trial history ----
    prev_role = np.concatenate([[""], role[:-1]])
    n_std_before = np.zeros(n_trials, dtype=int)     # standards in a row just before this trial
    count = 0
    for k in range(n_trials):
        n_std_before[k] = count
        count = count + 1 if role[k] == "standard" else 0

    # ---- sample index range of each window ----
    ss = lambda x: np.searchsorted(t, x, side="left")
    i_pre, i_on, i_off, i_post = ss(starts - pre_win), ss(starts), ss(stops), ss(stops + post_win)

    def window_stats(i0, i1):
        mean     = np.full((n_trials, n_roi), np.nan)
        nan_frac = np.full((n_trials, n_roi), np.nan)
        n_fin    = np.zeros((n_trials, n_roi), dtype=int)
        n_samp   = np.maximum(i1 - i0, 0)
        for k in range(n_trials):
            if n_samp[k] == 0:
                continue
            chunk = dff[i0[k]:i1[k], :]
            nan_frac[k] = np.isnan(chunk).mean(axis=0)
            fin = np.isfinite(chunk)
            n_fin[k] = fin.sum(axis=0)
            for r in range(n_roi):
                if n_fin[k, r] > 0:
                    mean[k, r] = chunk[fin[:, r], r].mean()
        return mean, n_samp, nan_frac, n_fin

    base, n_base, nan_base, fin_base = window_stats(i_pre, i_on)
    resp, n_resp, nan_resp, fin_resp = window_stats(i_on, i_off)
    post, n_post, nan_post, fin_post = window_stats(i_off, i_post)

    # ---- rule for relative and baseline analyses: both windows must be good ----
    enough = ((n_base >= min_points) & (n_resp >= min_points))[:, None] & np.ones((1, n_roi), bool)
    nan_ok = (nan_base <= max_nan_frac) & (nan_resp <= max_nan_frac)
    fin_ok = (fin_base >= min_points) & (fin_resp >= min_points)
    valid_rel = enough & nan_ok & fin_ok
    drop_reason = np.where(~enough, "too_few_points",
                  np.where(~nan_ok, "too_many_nans",
                  np.where(~fin_ok, "too_few_finite", "")))

    # ---- rule for absolute analyses: only the window itself must be good ----
    def valid_single_window(n_samp, n_fin):
        with np.errstate(divide="ignore", invalid="ignore"):
            fin_frac = n_fin / n_samp[:, None]
        return (n_samp[:, None] >= min_points) & (fin_frac >= min_finite_frac) & (n_fin >= min_points)

    valid_abs  = valid_single_window(n_resp, fin_resp)
    valid_post = valid_single_window(n_post, fin_post)

    rep = lambda a: np.repeat(a, n_roi)
    return pd.DataFrame({
        "block": block_id, "trial_id": rep(trial_id), "trial_in_block": rep(np.arange(n_trials)),
        "start_time": rep(starts), "stop_time": rep(stops),
        "orientation_deg": rep(ori), "contrast": rep(contrast),
        "role": rep(role), "condition": rep(condition),
        "prev_role": rep(prev_role), "n_std_before": rep(n_std_before),
        "in_block": rep(~np.isin(trial_id, list(not_in_block_ids))),
        "roi": np.tile(np.arange(n_roi), n_trials),
        "base": base.ravel(), "resp": resp.ravel(), "rel": (resp - base).ravel(), "post": post.ravel(),
        "n_base": rep(n_base), "n_resp": rep(n_resp), "n_post": rep(n_post),
        "valid_rel": valid_rel.ravel(), "valid_abs": valid_abs.ravel(), "valid_post": valid_post.ravel(),
        "drop_reason": drop_reason.ravel(),
    })


def ids_not_in_block(block_labeled):
    """Trials whose 'diameter' differs from the rest of the block (picked up from a neighbouring block)."""
    if "diameter" not in block_labeled.columns:
        return []
    d = block_labeled["diameter"]
    return list(block_labeled.index[d != d.mode().iloc[0]])


trial_table = pd.concat([
    build_trial_table(B["dff"], B["t"], B["stim"], block_id=b, standard_ori=B["standard_ori"],
                      not_in_block_ids=ids_not_in_block(B["labeled"]))
    for b, B in BLOCKS.items()
], ignore_index=True)
trial_table.insert(0, "dmd", DMD)
trial_table.insert(0, "mouse", MOUSE_ID)

print(f"trial_table: {len(trial_table):,} rows = trials x ROIs ({N_ROI} ROIs)")
_one = trial_table[trial_table["roi"] == EXAMPLE_ROI]
print(f"\nTrials per condition (counts shown for ROI {EXAMPLE_ROI}):")
display(_one.groupby(["block", "condition"]).agg(
    trials=("trial_id", "size"), kept_relative=("valid_rel", "sum"), kept_absolute=("valid_abs", "sum")))
print("Trials flagged as not belonging to their block:",
      sorted(trial_table.loc[~trial_table["in_block"], "trial_id"].unique().tolist()))

### Shared state on each trial: the global baseline

`global_base` is the mean baseline of all *other* ROIs on the same trial. It measures how active the whole imaging plane was just before the stimulus, without using the ROI itself.

In [ ]:
def add_global_baseline(table):
    table = table.copy()
    table["global_base"] = np.nan
    for b in sorted(table["block"].unique()):
        idx = table.index[table["block"] == b]
        sub = table.loc[idx].sort_values(["trial_in_block", "roi"])
        base = np.where(sub["valid_rel"], sub["base"], np.nan).reshape(-1, N_ROI)
        finite = np.isfinite(base)
        total = np.nansum(base, axis=1, keepdims=True)
        count = finite.sum(axis=1, keepdims=True)
        own = np.where(finite, base, 0.0)
        with np.errstate(divide="ignore", invalid="ignore"):
            loo = (total - own) / (count - finite)
        loo[(count - finite) <= 0] = np.nan
        table.loc[sub.index, "global_base"] = loo.ravel()
    return table


trial_table = add_global_baseline(trial_table)
print("global_base added. Example (Block 1, first standard trials, ROI 0):")
display(trial_table[(trial_table["block"] == 1) & (trial_table["roi"] == 0)]
        [["trial_in_block", "condition", "base", "global_base", "resp", "rel"]].head(5).round(4))

### Control trials: the same gratings in the orientation blocks

Outside the oddball blocks, gratings of many directions are shown in random order, so no grating is repeated or expected. These trials are the **control**: the same stimulus without repetition and without surprise.

Only trials with the same contrast, spatial frequency, temporal frequency and size as the first orientation block are used. `epoch` says which orientation block a trial comes from (`O1` before Block 1, `O2` between the blocks, `O3` after Block 2).

In [ ]:
def build_control_table():
    pos = np.arange(len(stim_table))
    ids = stim_table.index.to_numpy()

    def block_mask(start, end, labeled):
        return (pos >= start) & (pos <= end) & ~np.isin(ids, ids_not_in_block(labeled))

    in_b1 = block_mask(b1_start, b1_end, block1_labeled)
    in_b2 = block_mask(b2_start, b2_end, block2_labeled)
    mid1, mid2 = (b1_start + b1_end) / 2, (b2_start + b2_end) / 2
    epoch = np.select([in_b1, in_b2, pos < mid1, pos < mid2], ["B1", "B2", "O1", "O2"], default="O3")

    # same stimulus settings as the oddball blocks
    ref = block1_labeled[block1_labeled["contrast"] != 0.0]
    same = np.isin(epoch, ["O1", "O2", "O3"])
    for col in ["contrast", "spatial_frequency", "temporal_frequency"]:
        if col in stim_table.columns:
            same &= np.isclose(stim_table[col].to_numpy(float), float(ref[col].mode().iloc[0]))
    # same size as the first orientation block (leaves out receptive-field mapping trials)
    if "diameter" in stim_table.columns:
        d = stim_table["diameter"].to_numpy()
        same &= d == pd.Series(d[epoch == "O1"]).mode().iloc[0]

    sub = stim_table.loc[same, ["start_time", "stop_time", "contrast", "orientation_deg"]].copy()
    tab = build_trial_table(dmd1_dff, t1, sub, block_id=0, standard_ori=-999)
    direction = np.round(np.degrees(stim_table.loc[same, "orientation"].to_numpy(float)), 1)
    tab["epoch"] = tab["trial_id"].map(dict(zip(ids[same], epoch[same])))
    tab["direction_deg"] = tab["trial_id"].map(dict(zip(ids[same], direction)))

    # which control trials show exactly the gratings used in the oddball blocks
    both = pd.concat([block1_labeled, block2_labeled])
    both = both[both["contrast"] != 0.0]
    block_dirs = np.round(np.degrees(both.groupby("orientation_deg")["orientation"].median()), 1)
    tab["block_ori"] = tab["direction_deg"].map({v: int(k) for k, v in block_dirs.items()})
    tab["role"] = tab["condition"] = "control"
    return tab.drop(columns=["block", "prev_role", "n_std_before", "in_block", "trial_in_block"])


control_table = build_control_table()
_c0 = control_table[control_table["roi"] == 0]
print(f"control_table: {_c0['trial_id'].nunique()} trials in the orientation blocks, "
      f"{_c0['direction_deg'].nunique()} grating directions.")
print("\nControl trials that show the gratings of the oddball blocks (counts for one ROI):")
display(_c0.dropna(subset=["block_ori"]).astype({"block_ori": int}).groupby(["block_ori", "epoch"])
        .agg(trials=("trial_id", "size"), kept_relative=("valid_rel", "sum"), kept_absolute=("valid_abs", "sum")))

### Per-ROI means

`roi_means(block, condition, measure)` gives one value per ROI, averaged over the kept trials.

| measure | value | trials used |
|---|---|---|
| `"abs"` | absolute response | `valid_abs` |
| `"rel"` | relative response | `valid_rel` |
| `"base"` | baseline | `valid_rel` |
| `"resp_rel"` | absolute response, restricted to the relative-analysis trials | `valid_rel` |
| `"post"` | post-stimulus window | `valid_post` |

In [ ]:
MEASURES = {
    "abs":      ("resp", "valid_abs"),
    "rel":      ("rel",  "valid_rel"),
    "base":     ("base", "valid_rel"),
    "resp_rel": ("resp", "valid_rel"),
    "post":     ("post", "valid_post"),
}
_matrix_cache = {}


def condition_matrices(block, condition):
    """Trials x ROIs arrays for one block and condition, trials in time order."""
    key = (block, condition, USE_ONLY_IN_BLOCK_TRIALS)
    if key not in _matrix_cache:
        sub = trial_table[(trial_table["block"] == block) & (trial_table["condition"] == condition)]
        if USE_ONLY_IN_BLOCK_TRIALS:
            sub = sub[sub["in_block"]]
        if len(sub) == 0:
            raise ValueError(f"No trials for block {block}, condition '{condition}'.")
        sub = sub.sort_values(["trial_in_block", "roi"])
        n_trials = len(sub) // N_ROI
        mat = lambda col: sub[col].to_numpy().reshape(n_trials, N_ROI)
        out = {c: mat(c) for c in ["base", "resp", "rel", "post", "valid_rel", "valid_abs", "valid_post"]}
        for c in ["trial_id", "start_time", "stop_time", "prev_role", "n_std_before", "in_block"]:
            out[c] = mat(c)[:, 0]
        _matrix_cache[key] = out
    return _matrix_cache[key]


def roi_means(block, condition, measure):
    """Mean over kept trials for every ROI. Returns (values, number of kept trials)."""
    m = condition_matrices(block, condition)
    col, valid = MEASURES[measure]
    values = np.full(N_ROI, np.nan)
    n_kept = np.zeros(N_ROI, dtype=int)
    for r in range(N_ROI):
        v = m[col][m[valid][:, r], r]
        if v.size > 0:
            values[r] = v.mean()
            n_kept[r] = v.size
    return values, n_kept


# quick overview: mean across ROIs of each per-ROI mean
_rows = []
for _b in BLOCKS:
    for _c in ["standard"] + mismatch_conditions(_b):
        _row = {"block": _b, "condition": cond_label(_b, _c)}
        for _m in ["abs", "rel", "base"]:
            _v, _n = roi_means(_b, _c, _m)
            _row[f"{_m} (mean over ROIs)"] = np.nanmean(_v)
            _row[f"{_m} kept trials"] = int(np.median(_n))
        _rows.append(_row)
display(pd.DataFrame(_rows).round(4))

### Chance samples of standard trials

Mismatch conditions have about 30 trials and the standard has about 1,400. Many analyses below therefore ask: *what would 30 standard trials give by chance?*

`standard_samples(block, measure, n)` draws `n` standard trials at random, takes the per-ROI mean, and repeats this `N_DRAWS` times. Only standards that follow another standard are drawn, because oddballs and omissions also follow a standard.

In [ ]:
CONTROL_EPOCHS = ["O1", "O2", "O3"]      # which orientation blocks count as control
_sample_cache = {}


def control_means(ori_deg, measure, epochs=None):
    """Per-ROI mean of the control trials that show one of the oddball-block gratings."""
    col, valid = MEASURES[measure]
    sub = control_table[(control_table["block_ori"] == ori_deg)
                        & control_table["epoch"].isin(epochs or CONTROL_EPOCHS) & control_table[valid]]
    values, n = np.full(N_ROI, np.nan), np.zeros(N_ROI, int)
    for r, g in sub.groupby("roi"):
        values[r], n[r] = g[col].to_numpy().mean(), len(g)
    return values, n


def standard_pool(block, measure):
    """Trials x ROIs values of the standards that may be drawn (NaN where a trial is not usable)."""
    col, valid = MEASURES[measure]
    m = condition_matrices(block, "standard")
    rows = m["in_block"] & (m["prev_role"] == "standard") & m[valid].any(axis=1)
    return np.where(m[valid], m[col], np.nan)[rows]


def standard_samples(block, measure, n, n_draws=N_DRAWS, seed=RANDOM_SEED):
    """Per-ROI means of n randomly drawn standard trials. Returns an array (n_draws, N_ROI)."""
    key = (block, measure, int(n), n_draws, seed, USE_ONLY_IN_BLOCK_TRIALS)
    if key not in _sample_cache:
        pool = standard_pool(block, measure)
        rng = np.random.default_rng(seed)
        out = np.empty((n_draws, N_ROI))
        for k in range(n_draws):
            out[k] = nanmean_quiet(pool[rng.choice(pool.shape[0], size=min(int(n), pool.shape[0]), replace=False)], axis=0)
        _sample_cache[key] = out
    return _sample_cache[key]


_s = standard_samples(1, "abs", 30)
print(f"Example: 30-standard samples in Block 1, ROI {EXAMPLE_ROI}: "
      f"absolute response {np.percentile(_s[:, EXAMPLE_ROI], 2.5):.3f} to {np.percentile(_s[:, EXAMPLE_ROI], 97.5):.3f} "
      f"(all standards: {roi_means(1, 'standard', 'abs')[0][EXAMPLE_ROI]:.3f})")

## 12. Example trials and response measures

Single-trial traces for one ROI, and one annotated trial showing how the baseline, absolute and relative response are measured (Fig. 2).

In [ ]:
def get_trial(block, condition, k=0, roi=EXAMPLE_ROI):
    """Row of trial_table for the k-th trial (time order) of a condition, for one ROI.
    Trials picked up from a neighbouring block are skipped."""
    sub = trial_table[(trial_table["block"] == block) & (trial_table["condition"] == condition)
                      & (trial_table["roi"] == roi) & trial_table["in_block"]]
    return sub.sort_values("start_time").iloc[k]


def plot_trial_trace(ax, block, condition, k=0, roi=EXAMPLE_ROI, shade=True):
    """Absolute dF/F of one trial, from one window before onset to one window after offset."""
    row = get_trial(block, condition, k, roi)
    t, dff = BLOCKS[block]["t"], BLOCKS[block]["dff"]
    seg = (t >= row["start_time"] - PRE_WIN) & (t <= row["stop_time"] + POST_WIN)
    x, y = t[seg], dff[seg, roi]
    ok = np.isfinite(y)
    ax.plot(x[ok], y[ok], lw=1.8)
    if shade:
        ax.axvspan(row["start_time"], row["stop_time"], color="red", alpha=0.15)
    ax.axvline(row["start_time"], color="red", ls="--", alpha=0.6)
    ax.axvline(row["stop_time"],  color="red", ls="--", alpha=0.6)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Absolute dF/F")
    ax.set_title(f"ROI {roi}, Block {block}: {cond_label(block, condition)}, trial {k + 1}")
    clean_axes(ax)
    return row


def plot_example_trials(block, roi=EXAMPLE_ROI):
    conds = ["standard", "omission"] + oddball_conditions(block)
    fig, axs = plt.subplots(2, 2, figsize=(12, 6))
    for ax, c in zip(axs.ravel(), conds):
        plot_trial_trace(ax, block, c, k=0, roi=roi)
    fig.tight_layout()
    save_fig(fig, f"example_trials_b{block}")
    plt.show()


plot_example_trials(1)

In [ ]:
def plot_annotated_trial(block, condition="standard", k=0, roi=EXAMPLE_ROI):
    """One trial with the baseline and stimulus windows, their means, and the relative response."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    row = plot_trial_trace(ax, block, condition, k, roi, shade=False)
    start, stop = row["start_time"], row["stop_time"]
    base, resp, rel = row["base"], row["resp"], row["rel"]

    ax.axvspan(start - PRE_WIN, start, color="#cdbb8b", alpha=0.25, lw=0)
    ax.axvspan(start, stop,            color="#f4a6a6", alpha=0.25, lw=0)
    ax.hlines(base, start - PRE_WIN, start, colors="#7a6a3a", linestyles="--", lw=2)
    ax.hlines(resp, start, stop,            colors="#b14a4a", linestyles="--", lw=2)

    y0, y1 = ax.get_ylim()
    ax.text(start - PRE_WIN / 2, y1, "Pre-stimulus baseline", ha="center", va="top", fontsize=12, color="#6b5b2a")
    ax.text((start + stop) / 2,  y1, "Stimulus window",       ha="center", va="top", fontsize=12, color="#a33a3a")
    ax.text(1.02, 0.60, f"baseline mean = {base:.2f}", transform=ax.transAxes, color="#6b5b2a", fontsize=11)
    ax.text(1.02, 0.50, f"response mean = {resp:.2f}", transform=ax.transAxes, color="#a33a3a", fontsize=11)
    ax.text(1.02, 0.36, f"relative response =\nresponse mean − baseline mean\n= {rel:.2f}",
            transform=ax.transAxes, fontsize=11, va="top")
    ax.set_title("")
    save_fig(fig, f"example_trial_annotated_b{block}")
    plt.show()
    print(f"Baseline mean = {base:.4f}   Response mean = {resp:.4f}   Relative response = {rel:.4f}")


plot_annotated_trial(1, "standard", k=0)

### 12.3 One oddball and the standards around it

For any mismatch trial, this shows the standard before it, the mismatch trial, and the standard after it, each with its baseline, response and relative value. If two or more mismatch trials come in a row, all of them are shown.

Single trials of one ROI are very noisy, so this is a tool for looking at the data, not evidence by itself. Section 17 averages these triplets.

In [ ]:
def oddball_triplets(block):
    """One row per oddball/omission trial: the standard before it, the run of mismatch trials, the standard after."""
    bt = (trial_table[(trial_table["block"] == block) & (trial_table["roi"] == 0)]
          .sort_values("trial_in_block").reset_index(drop=True))
    role = bt["role"].to_numpy()
    rows = []
    for i in np.flatnonzero(role != "standard"):
        a = i
        while a - 1 >= 0 and role[a - 1] != "standard":
            a -= 1
        z = i
        while z + 1 < len(bt) and role[z + 1] != "standard":
            z += 1
        rows.append({"condition": bt.loc[i, "condition"], "trial": int(bt.loc[i, "trial_in_block"]),
                     "pre": int(bt.loc[a - 1, "trial_in_block"]) if a - 1 >= 0 else None,
                     "run": [int(v) for v in bt.loc[a:z, "trial_in_block"]],
                     "post": int(bt.loc[z + 1, "trial_in_block"]) if z + 1 < len(bt) else None})
    out = pd.DataFrame(rows)
    out["occurrence"] = out.groupby("condition").cumcount() + 1
    return out


TRIPLETS = {b: oddball_triplets(b) for b in BLOCKS}
for _b, _tr in TRIPLETS.items():
    _multi = int((_tr["run"].apply(len) > 1).sum())
    print(f"Block {_b}: " + ", ".join(f"{cond_label(_b, c)} × {n}" for c, n in _tr["condition"].value_counts().items())
          + f"; {_multi} mismatch trials sit in a run of two or more.")


def plot_oddball_triplet(block, condition, occurrence=1, roi=EXAMPLE_ROI, save=False):
    """occurrence counts from 1. roi can be a number, or None for the mean over all ROIs."""
    tr = TRIPLETS[block]
    sel = tr[(tr["condition"] == condition) & (tr["occurrence"] == occurrence)]
    if len(sel) == 0:
        n_have = int((tr["condition"] == condition).sum())
        print(f"Block {block} has {n_have} trials of '{condition}'. "
              f"Conditions in this block: {mismatch_conditions(block)}.")
        return
    sel = sel.iloc[0]
    trials = [sel["pre"]] + list(sel["run"]) + [sel["post"]]
    names = ["Standard before"] + [None] * len(sel["run"]) + ["Standard after"]

    bt = trial_table[trial_table["block"] == block]
    t, dff = BLOCKS[block]["t"], BLOCKS[block]["dff"]
    fig, axs = plt.subplots(1, len(trials), figsize=(4.6 * len(trials), 4), sharey=True, squeeze=False)
    for ax, k, name in zip(axs[0], trials, names):
        if k is None or pd.isna(k):
            ax.set_title("no standard on this side")
            ax.axis("off")
            continue
        k = int(k)
        rows = bt[bt["trial_in_block"] == k]
        info = rows.iloc[0]
        cond = info["condition"]
        seg = (t >= info["start_time"] - PRE_WIN) & (t <= info["stop_time"] + POST_WIN)
        x = t[seg] - info["start_time"]
        if roi is None:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                y = np.nanmean(dff[seg, :], axis=1)
            vals = rows[rows["valid_rel"]][["base", "resp", "rel"]].mean()
        else:
            y = dff[seg, roi]
            vals = rows[rows["roi"] == roi].iloc[0][["base", "resp", "rel"]]
        ok = np.isfinite(y)
        color = cond_color(block, cond)
        ax.plot(x[ok], y[ok], lw=1.6, color=color)
        dur = info["stop_time"] - info["start_time"]
        ax.axvspan(0, dur, color="red" if cond != "omission" else "0.5", alpha=0.10)
        ax.axvline(0, color="k", ls="--", lw=0.8)
        if np.isfinite(vals["base"]):
            ax.hlines(vals["base"], -PRE_WIN, 0, colors="#7a6a3a", linestyles="--", lw=1.5)
        if np.isfinite(vals["resp"]):
            ax.hlines(vals["resp"], 0, dur, colors="#b14a4a", linestyles="--", lw=1.5)
        selected = (k == sel["trial"])
        title = name or (cond_label(block, cond) + (f" (#{occurrence})" if selected else ""))
        ax.set_title(f"{title}\nbaseline {vals['base']:.2f}, response {vals['resp']:.2f}, relative {vals['rel']:+.2f}",
                     fontsize=11, fontweight="bold" if selected else "normal")
        ax.set_xlabel("Time relative to onset (s)")
        clean_axes(ax)
    axs[0, 0].set_ylabel("Absolute dF/F" + (" (mean of all ROIs)" if roi is None else f" (ROI {roi})"))
    fig.suptitle(f"Block {block}: {cond_label(block, condition)}, occurrence {occurrence} "
                 f"(trial {int(sel['trial']) + 1} of the block)", y=1.04)
    fig.tight_layout()
    if save:
        save_fig(fig, f"triplet_b{block}_{condition}_{occurrence}_roi{'mean' if roi is None else roi}")
    plt.show()


# example: the first oddball of Block 1 and its neighbours.
# Change the numbers to look at any other one, e.g. plot_oddball_triplet(1, "oddball_45", 12, roi=3)
plot_oddball_triplet(1, oddball_conditions(1)[0], occurrence=1, roi=EXAMPLE_ROI)

## 13. Peri-stimulus time histograms (PSTH)

Trial-averaged absolute dF/F aligned to stimulus onset. Each trial is interpolated onto a common time axis; trials with more than half of their samples missing are left out.

### 13.1 Example ROI: standard, oddballs and omission (Fig. 3)

In [ ]:
def psth(block, condition, roi, pre=PRE_WIN, post=POST_WIN, max_nan_frac=MAX_NAN_FRAC):
    """Mean aligned trace of one ROI for one condition.
    Returns common_t, mean trace, SEM trace, number of trials used, stimulus duration."""
    B = BLOCKS[block]
    t, dff, stim = B["t"], B["dff"], B["stim"].sort_values("start_time")

    first_std = stim[(stim["orientation_deg"] == B["standard_ori"]) & (stim["contrast"] != 0.0)].iloc[0]
    stim_dur = float(first_std["stop_time"]) - float(first_std["start_time"])
    dt = np.median(np.diff(t))
    common_t = np.arange(-pre, stim_dur + post + dt, dt)

    starts = condition_matrices(block, condition)["start_time"]
    traces = []
    for t0 in starts:
        i0 = np.searchsorted(t, t0 - pre, side="left")
        i1 = np.searchsorted(t, t0 + stim_dur + post, side="right")
        x, y = t[i0:i1] - t0, dff[i0:i1, roi]
        if x.size < 3 or np.isnan(y).mean() > max_nan_frac:
            continue
        ok = np.isfinite(y)
        if ok.sum() < 3:
            continue
        traces.append(np.interp(common_t, x[ok], y[ok]))

    if len(traces) == 0:
        nan = np.full_like(common_t, np.nan, dtype=float)
        return common_t, nan, nan, 0, stim_dur
    traces = np.vstack(traces)
    mean = np.nanmean(traces, axis=0)
    sem = np.nanstd(traces, axis=0) / np.sqrt(traces.shape[0])
    return common_t, mean, sem, traces.shape[0], stim_dur


def stim_window_mean(trace, common_t, stim_dur):
    m = (common_t >= 0.0) & (common_t < stim_dur)
    return float(np.nanmean(trace[m])) if np.any(m) else np.nan


def plot_psth_conditions(block, roi=EXAMPLE_ROI):
    fig, ax = plt.subplots(figsize=(12, 4))
    lines = ["Stimulus-window mean dF/F"]
    results = {}
    for c in ["standard", "omission"] + oddball_conditions(block)[::-1]:
        common_t, mean, sem, n, stim_dur = psth(block, c, roi)
        ax.plot(common_t, mean, lw=2.2 if c == "standard" else 1.8, color=cond_color(block, c),
                label=f"{cond_label(block, c)} (n={n})")
        results[c] = dict(n=n, window_mean=stim_window_mean(mean, common_t, stim_dur))
        lines.append(f"{cond_label(block, c)} = {results[c]['window_mean']:.3f}")
    ax.axvspan(0, stim_dur, color="red", alpha=0.08)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_xlabel("Time relative to onset (s)")
    ax.set_ylabel(f"dF/F (ROI {roi})")
    ax.set_title(f"Block {block}: trial-averaged response, ROI {roi}")
    ax.text(1.02, 0.10, "\n".join(lines), transform=ax.transAxes, fontsize=10, va="bottom", ha="left")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.72), frameon=False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.grid(False)
    save_fig(fig, f"psth_conditions_b{block}")
    plt.show()
    print(f"Block {block}, ROI {roi}:")
    for c, res in results.items():
        print(f"  {cond_label(block, c):13s}: n = {res['n']:4d}, stimulus-window mean = {res['window_mean']:.4f}")
    return results


psth_results = {b: plot_psth_conditions(b) for b in BLOCKS}

### 13.2 Every ROI, with a chance band for the standard

The black standard trace averages about 1,400 trials; the oddball and omission traces only about 30. The grey band shows what a 30-trial average of the **standard** looks like by chance: 30 standard trials are drawn at random and averaged, 1,000 times, and the band is the middle 95% at each time point.

The band is a per-time-point range, so a trace will leave it at about 5% of time points by chance. The p-value in each title is the safer summary: it compares the stimulus-window mean of a condition with standard samples of the same trial count.

In [ ]:
N_SUB        = 30      # trials per random standard sample (same size as the oddball conditions)
N_BAND_DRAWS = N_DRAWS # number of random samples
_psth_trials_cache = {}
_band_cache = {}


def psth_trials(block, condition, roi, pre=PRE_WIN, post=POST_WIN, max_nan_frac=MAX_NAN_FRAC):
    """Aligned single-trial traces of one ROI (same trial rules as psth()).
    Returns a dict: common_t, traces (kept trials x time), prev_role of the kept trials, stim_dur."""
    key = (block, condition, roi, pre, post, max_nan_frac, USE_ONLY_IN_BLOCK_TRIALS)
    if key in _psth_trials_cache:
        return _psth_trials_cache[key]

    B = BLOCKS[block]
    t, dff, stim = B["t"], B["dff"], B["stim"].sort_values("start_time")
    first_std = stim[(stim["orientation_deg"] == B["standard_ori"]) & (stim["contrast"] != 0.0)].iloc[0]
    stim_dur = float(first_std["stop_time"]) - float(first_std["start_time"])
    dt = np.median(np.diff(t))
    common_t = np.arange(-pre, stim_dur + post + dt, dt)

    m = condition_matrices(block, condition)
    traces, kept = [], []
    for k, t0 in enumerate(m["start_time"]):
        i0 = np.searchsorted(t, t0 - pre, side="left")
        i1 = np.searchsorted(t, t0 + stim_dur + post, side="right")
        x, y = t[i0:i1] - t0, dff[i0:i1, roi]
        if x.size < 3 or np.isnan(y).mean() > max_nan_frac:
            continue
        ok = np.isfinite(y)
        if ok.sum() < 3:
            continue
        traces.append(np.interp(common_t, x[ok], y[ok]))
        kept.append(k)

    out = dict(common_t=common_t, stim_dur=stim_dur,
               traces=np.vstack(traces) if traces else np.empty((0, common_t.size)),
               prev_role=m["prev_role"][kept] if kept else np.array([], dtype=object))
    _psth_trials_cache[key] = out
    return out


def standard_sample_band(block, roi, n_sub=N_SUB, n_draws=N_BAND_DRAWS,
                         only_after_standard=True, seed=RANDOM_SEED):
    """Averages of n_sub randomly drawn standard trials, repeated n_draws times.
    only_after_standard=True draws only standards that follow another standard,
    because oddballs and omissions also follow a standard.
    Returns: lo, hi (2.5 and 97.5 percentile at each time point), window_means (one per draw), pool size."""
    key = (block, roi, n_sub, n_draws, only_after_standard, seed, USE_ONLY_IN_BLOCK_TRIALS)
    if key in _band_cache:
        return _band_cache[key]
    d = psth_trials(block, "standard", roi)
    tr = d["traces"]
    if only_after_standard:
        tr = tr[d["prev_role"] == "standard"]
    n_sub = min(n_sub, tr.shape[0])
    rng = np.random.default_rng(seed + 1000 * block + roi)
    means = np.empty((n_draws, tr.shape[1]))
    for k in range(n_draws):
        means[k] = tr[rng.choice(tr.shape[0], size=n_sub, replace=False)].mean(axis=0)
    lo, hi = np.percentile(means, [2.5, 97.5], axis=0)
    win = (d["common_t"] >= 0.0) & (d["common_t"] < d["stim_dur"])
    _band_cache[key] = (lo, hi, means[:, win].mean(axis=1), tr.shape[0])
    return _band_cache[key]


def plot_psth_with_band(block, roi, n_sub=N_SUB, n_draws=N_BAND_DRAWS, save=False, show=True):
    """PSTH of one ROI with the chance band of the standard. Returns the stimulus-window results.
    The band uses n_sub trials per sample. Each p-value uses samples of that condition's own trial count."""
    lo, hi, win_means, n_pool = standard_sample_band(block, roi, n_sub, n_draws)
    w_lo, w_hi = np.percentile(win_means, [2.5, 97.5])

    fig, ax = plt.subplots(figsize=(12, 4))
    lines = ["Stimulus-window mean dF/F", f"{n_sub}-standard samples: {w_lo:.3f} to {w_hi:.3f}"]
    results = {}
    for c in ["standard", "omission"] + oddball_conditions(block)[::-1]:
        d = psth_trials(block, c, roi)
        common_t, stim_dur, n = d["common_t"], d["stim_dur"], d["traces"].shape[0]
        mean = d["traces"].mean(axis=0) if n else np.full(common_t.size, np.nan)
        wm = stim_window_mean(mean, common_t, stim_dur)
        if c == "standard":
            ax.fill_between(common_t, lo, hi, color="0.6", alpha=0.35, lw=0,
                            label=f"95% range of {n_sub}-standard averages")
            results[c] = dict(n=n, window_mean=wm, p_vs_standard_samples=np.nan)
            lines.append(f"{cond_label(block, c)} = {wm:.3f}")
        else:
            # p-value: compare with standard samples of the same size as this condition
            chance = standard_sample_band(block, roi, n_sub=n, n_draws=n_draws)[2] if n >= 2 else win_means
            p = chance_p(wm, chance)
            c_lo, c_hi = np.percentile(chance, [2.5, 97.5])
            results[c] = dict(n=n, window_mean=wm, p_vs_standard_samples=p, chance_lo=c_lo, chance_hi=c_hi)
            lines.append(f"{cond_label(block, c)} = {wm:.3f} (p = {p:.3f})")
        ax.plot(common_t, mean, lw=2.2 if c == "standard" else 1.8, color=cond_color(block, c),
                label=f"{cond_label(block, c)} (n={n})")

    ax.axvspan(0, stim_dur, color="red", alpha=0.08)
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_xlabel("Time relative to onset (s)")
    ax.set_ylabel(f"dF/F (ROI {roi})")
    ax.set_title(f"Block {block}: trial-averaged response, ROI {roi}")
    ax.text(1.02, 0.02, "\n".join(lines), transform=ax.transAxes, fontsize=10, va="bottom", ha="left")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.72), frameon=False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.grid(False)
    if save:
        save_fig(fig, f"psth_with_band_b{block}_roi{roi}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return results


def plot_psth_grid(block, n_sub=N_SUB, n_cols=3, save=True):
    n_rows = int(np.ceil(N_ROI / n_cols))
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(6.2 * n_cols, 3.0 * n_rows), sharex=True, squeeze=False)
    conds = ["standard", "omission"] + oddball_conditions(block)[::-1]
    for roi in range(N_ROI):
        ax = axs[roi // n_cols, roi % n_cols]
        lo, hi, _, _ = standard_sample_band(block, roi, n_sub)
        p_txt = []
        for c in conds:
            d = psth_trials(block, c, roi)
            n = d["traces"].shape[0]
            if n == 0:
                continue
            mean = d["traces"].mean(axis=0)
            if c == "standard":
                ax.fill_between(d["common_t"], lo, hi, color="0.6", alpha=0.35, lw=0,
                                label=f"95% range of {n_sub}-standard averages")
            else:
                chance = standard_sample_band(block, roi, n_sub=n)[2]
                wm = stim_window_mean(mean, d["common_t"], d["stim_dur"])
                p_txt.append(f"{cond_label(block, c).replace('Oddball ', '')} p={chance_p(wm, chance):.2f}")
            ax.plot(d["common_t"], mean, lw=1.8 if c == "standard" else 1.1, color=cond_color(block, c),
                    label=cond_label(block, c))
        ax.axvspan(0, d["stim_dur"], color="red", alpha=0.06)
        ax.axvline(0, color="k", ls="--", lw=0.8)
        ax.set_title(f"ROI {roi}   ({', '.join(p_txt)})", fontsize=10)
        ax.tick_params(labelsize=9)
        clean_axes(ax)
    for k in range(N_ROI, n_rows * n_cols):
        axs[k // n_cols, k % n_cols].axis("off")
    for ax in axs[-1]:
        ax.set_xlabel("Time relative to onset (s)", fontsize=10)
    for ax in axs[:, 0]:
        ax.set_ylabel("dF/F", fontsize=10)
    handles, labels = axs[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False,
               bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.suptitle(f"Block {block}: trial-averaged response of every ROI "
                 f"(p = stimulus-window mean against standard samples)", y=1.03, fontsize=12)
    fig.tight_layout()
    if save:
        save_fig(fig, f"psth_grid_b{block}")
    plt.show()


for _b in BLOCKS:
    plot_psth_grid(_b)

The same comparison as a table, one row per ROI and condition.

In [ ]:
_rows = []
for _b in BLOCKS:
    for _roi in range(N_ROI):
        _res = plot_psth_with_band(_b, _roi, show=False)
        for _c in mismatch_conditions(_b):
            _rows.append({"block": _b, "roi": _roi, "condition": cond_label(_b, _c),
                          "trials": _res[_c]["n"],
                          "window mean": _res[_c]["window_mean"],
                          "standard (all trials)": _res["standard"]["window_mean"],
                          "chance 2.5%": _res[_c]["chance_lo"], "chance 97.5%": _res[_c]["chance_hi"],
                          "p": _res[_c]["p_vs_standard_samples"]})
psth_band_table = pd.DataFrame(_rows)
psth_band_table["outside chance range"] = psth_band_table["p"] < 0.05
display(psth_band_table.round(4))

_n_tests = len(psth_band_table)
print(f"{int(psth_band_table['outside chance range'].sum())} of {_n_tests} ROI-condition pairs are outside the 95% chance range; "
      f"about {0.05 * _n_tests:.0f} would be expected by chance alone.")
display(psth_band_table.groupby(["block", "condition"])["outside chance range"].agg(["sum", "count"])
        .rename(columns={"sum": "ROIs outside chance range", "count": "ROIs tested"}))

## 14. Is each ROI driven by the grating, and does context change it?

Each grating of the oddball blocks is seen in up to three contexts:

- **control:** in the orientation blocks, where it is neither repeated nor unexpected
- **standard:** repeated about 1,400 times in an oddball block
- **oddball:** rare and unexpected in an oddball block

Two comparisons follow from this:

- **control vs standard** shows the effect of repetition (adaptation).
- **oddball vs control** shows the effect of being unexpected, beyond not being repeated.

### 14.1 The three contexts, ROI by ROI

The relative response is the fairer row here: control trials come from other parts of the session, and slow changes in overall level affect the absolute response.

In [ ]:
def contexts_for(ori_deg):
    """[(label, function measure -> (values, n))] for one grating: control, then each block where it appears."""
    out = [("control", lambda measure: control_means(ori_deg, measure))]
    for b in BLOCKS:
        if BLOCKS[b]["standard_ori"] == ori_deg:
            out.append((f"standard (B{b})", lambda measure, b=b: roi_means(b, "standard", measure)))
        elif f"oddball_{ori_deg}" in oddball_conditions(b):
            out.append((f"oddball (B{b})", lambda measure, b=b: roi_means(b, f"oddball_{ori_deg}", measure)))
    return sorted(out, key=lambda x: ("control", "standard", "oddball").index(x[0].split(" ")[0]))


def plot_context_comparison():
    oris = sorted(int(o) for o in control_table["block_ori"].dropna().unique())
    fig, axs = plt.subplots(2, len(oris), figsize=(5.2 * len(oris), 8.5), squeeze=False)
    rows = []
    for i, (measure, unit) in enumerate([("rel", "Relative response (resp − baseline)"),
                                         ("abs", "Absolute response (dF/F)")]):
        for j, ori in enumerate(oris):
            ax = axs[i, j]
            ctx = contexts_for(ori)
            vals = {lab: get(measure) for lab, get in ctx}
            Y = np.array([vals[lab][0] for lab, _ in ctx])                 # contexts x ROIs
            xs = np.arange(len(ctx))
            ax.plot(xs, Y, color="0.75", lw=1, marker="o", ms=3)
            mean, err = np.nanmean(Y, axis=1), sem(Y, axis=1)
            ax.errorbar(xs, mean, yerr=err, color="k", lw=2.2, marker="o", capsize=4, zorder=5)
            if measure == "rel":
                ax.axhline(0, color="0.5", ls="--", lw=1)
            ax.set_xticks(xs)
            ax.set_xticklabels([lab for lab, _ in ctx])
            ax.set_xlim(-0.4, len(ctx) - 0.6)
            ax.set_title(f"{ori}° grating")
            if j == 0:
                ax.set_ylabel(unit)
            clean_axes(ax)
            for k, (lab, _) in enumerate(ctx):
                rows.append({"measure": measure, "grating": ori, "context": lab,
                             "trials (median over ROIs)": int(np.median(vals[lab][1])),
                             "mean over ROIs": mean[k], "SEM over ROIs": err[k],
                             "difference from control": np.nan if lab == "control" else float(np.nanmean(Y[k] - Y[0])),
                             "p vs control (paired, ROIs)": np.nan if lab == "control" else paired_p(Y[k], Y[0])})
    fig.suptitle("Each grey line is one ROI; black is the mean ± SEM over ROIs", y=1.0)
    fig.tight_layout()
    save_fig(fig, "context_comparison")
    plt.show()
    return pd.DataFrame(rows)


context_table = plot_context_comparison()
display(context_table.round(4))

### 14.2 Time course: the standard against the same grating as control

- **Black:** the standard, all trials.
- **Green:** the same grating in the orientation blocks.
- **Grey band:** the middle 95% of standard averages that use as many trials as the control.

Every trial is shown relative to its own pre-stimulus mean. `evoked` is the stimulus-window mean minus the pre-stimulus mean.

In [ ]:
_control_psth_cache = {}


def control_psth_data(block, roi, n_draws=N_DRAWS, seed=RANDOM_SEED):
    """Aligned, baseline-subtracted traces of the standard and of the same grating as control, for one ROI."""
    key = (block, roi, n_draws, seed, tuple(CONTROL_EPOCHS), USE_ONLY_IN_BLOCK_TRIALS)
    if key in _control_psth_cache:
        return _control_psth_cache[key]
    ori = BLOCKS[block]["standard_ori"]
    std = condition_matrices(block, "standard")
    std_starts = std["start_time"][std["in_block"] & (std["prev_role"] == "standard")]
    ctl = control_table[(control_table["block_ori"] == ori) & (control_table["roi"] == 0)
                        & control_table["epoch"].isin(CONTROL_EPOCHS)]
    ctl_starts = np.sort(ctl["start_time"].to_numpy())
    stim_dur = float(np.median(std["stop_time"] - std["start_time"]))
    common_t = np.arange(-PRE_WIN, stim_dur + POST_WIN, np.median(np.diff(t1)))
    pre_w, stim_w = common_t < 0, (common_t >= 0) & (common_t < stim_dur)

    def aligned(starts):
        traces = []
        for t0 in starts:
            i0 = np.searchsorted(t1, t0 - PRE_WIN, side="left")
            i1 = np.searchsorted(t1, t0 + stim_dur + POST_WIN, side="right")
            x, y = t1[i0:i1] - t0, dmd1_dff[i0:i1, roi]
            if x.size < 3 or np.isnan(y).mean() > MAX_NAN_FRAC:
                continue
            ok = np.isfinite(y)
            if ok.sum() >= 3:
                traces.append(np.interp(common_t, x[ok], y[ok]))
        tr = np.vstack(traces) if traces else np.empty((0, common_t.size))
        return tr - tr[:, pre_w].mean(axis=1, keepdims=True)       # relative to each trial's own pre-stimulus mean

    S, C = aligned(std_starts), aligned(ctl_starts)
    out = dict(common_t=common_t, stim_dur=stim_dur, n_std=len(S), n_ctl=len(C), ori=ori)
    if len(S) and len(C):
        rng = np.random.default_rng(seed + 1000 * block + roi)
        n = min(len(C), len(S))
        draws = np.array([S[rng.choice(len(S), n, replace=False)].mean(axis=0) for _ in range(n_draws)])
        evoked = lambda trace: trace[..., stim_w].mean(axis=-1)      # pre-stimulus mean is already zero
        out.update(std_mean=S.mean(axis=0), ctl_mean=C.mean(axis=0),
                   lo=np.percentile(draws, 2.5, axis=0), hi=np.percentile(draws, 97.5, axis=0),
                   evoked_std=float(evoked(S.mean(axis=0))), evoked_ctl=float(evoked(C.mean(axis=0))),
                   p=chance_p(float(evoked(C.mean(axis=0))), evoked(draws)))
    _control_psth_cache[key] = out
    return out


def plot_control_psth_grid(block, n_cols=3):
    n_rows = int(np.ceil(N_ROI / n_cols))
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(6.2 * n_cols, 3.0 * n_rows), sharex=True, squeeze=False)
    rows = []
    for roi in range(N_ROI):
        ax = axs[roi // n_cols, roi % n_cols]
        d = control_psth_data(block, roi)
        if "std_mean" not in d:
            ax.set_title(f"ROI {roi}: no usable trials", fontsize=10)
            continue
        ax.fill_between(d["common_t"], d["lo"], d["hi"], color="0.6", alpha=0.35, lw=0,
                        label="95% range of standard averages (same trial count)")
        ax.plot(d["common_t"], d["std_mean"], color="k", lw=1.8, label=f"Standard {d['ori']}° in Block {block}")
        ax.plot(d["common_t"], d["ctl_mean"], color="tab:green", lw=1.4,
                label=f"Same {d['ori']}° grating, orientation blocks")
        ax.axvspan(0, d["stim_dur"], color="red", alpha=0.06)
        ax.axvline(0, color="k", ls="--", lw=0.8)
        ax.set_title(f"ROI {roi}  (evoked: control {d['evoked_ctl']:+.3f}, standard {d['evoked_std']:+.3f}, "
                     f"p={d['p']:.3f})", fontsize=10)
        ax.tick_params(labelsize=9)
        clean_axes(ax)
        rows.append({"block": block, "roi": roi, "control trials": d["n_ctl"], "standard trials": d["n_std"],
                     "evoked, control": d["evoked_ctl"], "evoked, standard": d["evoked_std"],
                     "p (control vs standard samples)": d["p"]})
    for k in range(N_ROI, n_rows * n_cols):
        axs[k // n_cols, k % n_cols].axis("off")
    for ax in axs[-1]:
        ax.set_xlabel("Time relative to onset (s)", fontsize=10)
    for ax in axs[:, 0]:
        ax.set_ylabel("dF/F minus pre-stimulus mean", fontsize=10)
    handles, labels = axs[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.suptitle(f"Block {block}: standard against the same grating when it is not repeated", y=1.03, fontsize=12)
    fig.tight_layout()
    save_fig(fig, f"control_psth_grid_b{block}")
    plt.show()
    return pd.DataFrame(rows)


control_psth_table = pd.concat([plot_control_psth_grid(b) for b in BLOCKS], ignore_index=True)
display(control_psth_table.round(4))
print(f"{int((control_psth_table['p (control vs standard samples)'] < 0.05).sum())} of {len(control_psth_table)} "
      f"ROI-block pairs have a control response outside the standard's chance range; "
      f"about {0.05 * len(control_psth_table):.0f} would be expected by chance alone.")

### 14.3 Same grating, different role (role switch between the blocks)

The 0° and 90° gratings swap roles between the blocks, so each is seen once as standard and once as oddball.

- **Role effect:** response as oddball minus response as standard, for the same grating and the same ROI.
- **Drift control:** the 45° grating is an oddball in both blocks, so its change between the blocks estimates how much responses drift for reasons other than role.
- **Corrected role effect:** role effect minus the drift of the 45° grating in the same direction of time.

In [ ]:
def role_switch_table():
    s1, s2 = BLOCKS[1]["standard_ori"], BLOCKS[2]["standard_ori"]
    rows, per_roi = [], {}
    for measure in ["rel", "abs"]:
        drift_45 = roi_means(2, "oddball_45", measure)[0] - roi_means(1, "oddball_45", measure)[0]   # B2 − B1
        effects = {
            f"{s1}°: oddball (B2) − standard (B1)": (roi_means(2, f"oddball_{s1}", measure)[0]
                                                    - roi_means(1, "standard", measure)[0], +1),
            f"{s2}°: oddball (B1) − standard (B2)": (roi_means(1, f"oddball_{s2}", measure)[0]
                                                    - roi_means(2, "standard", measure)[0], -1),
        }
        rows.append({"measure": measure, "contrast": "45°: oddball (B2) − oddball (B1)  [drift control]",
                     "mean over ROIs": np.nanmean(drift_45), "SEM over ROIs": sem(drift_45),
                     "ROIs above 0": int(np.sum(drift_45 > 0)), "p (paired, ROIs)": paired_p(drift_45)})
        for name, (eff, sign) in effects.items():
            corrected = eff - sign * drift_45
            per_roi[(measure, name)] = corrected
            for label, v in [(name, eff), (name + ", drift-corrected", corrected)]:
                rows.append({"measure": measure, "contrast": label,
                             "mean over ROIs": np.nanmean(v), "SEM over ROIs": sem(v),
                             "ROIs above 0": int(np.sum(v > 0)), "p (paired, ROIs)": paired_p(v)})
    return pd.DataFrame(rows), per_roi


role_table, role_effect_per_roi = role_switch_table()
display(role_table.round(4))

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, measure in zip(axs, ["rel", "abs"]):
    items = [(k[1], v) for k, v in role_effect_per_roi.items() if k[0] == measure]
    rng = np.random.default_rng(RANDOM_SEED)
    for i, (name, v) in enumerate(items):
        ax.scatter(i + rng.uniform(-0.1, 0.1, v.size), v, color="0.5", s=28)
        ax.errorbar(i + 0.25, np.nanmean(v), yerr=sem(v), fmt="o", color="k", capsize=4)
    ax.axhline(0, color="0.5", ls="--", lw=1)
    ax.set_xticks(range(len(items)))
    ax.set_xticklabels([n.split(":")[0] + " grating" for n, _ in items])
    ax.set_xlim(-0.5, len(items) - 0.3)
    ax.set_ylabel("Oddball − standard, drift-corrected")
    ax.set_title("Relative response" if measure == "rel" else "Absolute response")
    clean_axes(ax)
fig.suptitle("Role effect per ROI (dots) with mean ± SEM (black)", y=1.02)
fig.tight_layout()
save_fig(fig, "role_switch")
plt.show()

### 14.4 Orientation tuning in the orientation blocks, and whether oddball responses follow it

The orientation blocks show many grating directions. For each ROI, the tuning curve is its evoked response (relative response) to each direction.

- **Is the ROI tuned at all?** A one-way test across directions, per ROI.
- **Do oddball responses follow tuning?** If an ROI prefers 90° over 45° in the control trials, does it also respond more to the 90° oddball than to the 45° oddball?

In [ ]:
def tuning_curves(measure="rel"):
    col, valid = MEASURES[measure]
    sub = control_table[control_table[valid] & control_table["epoch"].isin(CONTROL_EPOCHS)]
    g = sub.groupby(["roi", "direction_deg"])[col]
    return g.mean().unstack(), g.sem().unstack(), g.size().unstack()


tuning_mean, tuning_sem, tuning_n = tuning_curves("rel")
_block_dirs = (control_table.dropna(subset=["block_ori"]).groupby("block_ori")["direction_deg"].first())

n_cols = 3
n_rows = int(np.ceil(N_ROI / n_cols))
fig, axs = plt.subplots(n_rows, n_cols, figsize=(5.4 * n_cols, 2.8 * n_rows), sharex=True, squeeze=False)
_tuning_rows = []
for roi in range(N_ROI):
    ax = axs[roi // n_cols, roi % n_cols]
    dirs = tuning_mean.columns.to_numpy(float)
    ax.errorbar(dirs, tuning_mean.loc[roi], yerr=tuning_sem.loc[roi], color="k", marker="o", ms=3, lw=1.2, capsize=2)
    for ori, d in _block_dirs.items():
        ax.axvline(d, color="tab:green", ls=":", lw=1)
    ax.axhline(0, color="0.5", ls="--", lw=0.8)
    # one-way test across directions
    _sub = control_table[(control_table["roi"] == roi) & control_table["valid_rel"]
                         & control_table["epoch"].isin(CONTROL_EPOCHS)]
    _groups = [g["rel"].to_numpy() for _, g in _sub.groupby("direction_deg") if len(g) >= 3]
    p_tuned = float(sp_stats.f_oneway(*_groups).pvalue) if (sp_stats is not None and len(_groups) >= 2) else np.nan
    ax.set_title(f"ROI {roi}  (tuning test p = {p_tuned:.3f})", fontsize=10)
    ax.tick_params(labelsize=9)
    clean_axes(ax)
    _tuning_rows.append({"roi": roi, "preferred direction": float(tuning_mean.loc[roi].idxmax()),
                         "largest evoked response": float(tuning_mean.loc[roi].max()),
                         "smallest evoked response": float(tuning_mean.loc[roi].min()),
                         "tuning test p": p_tuned})
for k in range(N_ROI, n_rows * n_cols):
    axs[k // n_cols, k % n_cols].axis("off")
for ax in axs[-1]:
    ax.set_xlabel("Grating direction (deg)", fontsize=10)
for ax in axs[:, 0]:
    ax.set_ylabel("Evoked response", fontsize=10)
fig.suptitle("Tuning curves from the orientation blocks (mean ± SEM over trials; "
             "green lines: gratings used in the oddball blocks)", y=1.01, fontsize=12)
fig.tight_layout()
save_fig(fig, "tuning_curves")
plt.show()

tuning_table = pd.DataFrame(_tuning_rows)
display(tuning_table.round(4))
print(f"{int((tuning_table['tuning test p'] < 0.05).sum())} of {N_ROI} ROIs show tuning at p < 0.05 "
      f"(about {0.05 * N_ROI:.1f} expected by chance); median trials per direction: {int(np.nanmedian(tuning_n.to_numpy()))}.")

In [ ]:
# Do oddball responses follow the tuning measured in the control trials?
fig, axs = plt.subplots(1, len(BLOCKS), figsize=(5.5 * len(BLOCKS), 5), squeeze=False)
tuning_vs_oddball = {}
for ax, b in zip(axs[0], BLOCKS):
    cA, cB = oddball_conditions(b)
    oA, oB = int(cA.split("_")[1]), int(cB.split("_")[1])
    x = control_means(oA, "rel")[0] - control_means(oB, "rel")[0]          # tuning preference in control
    y = roi_means(b, cA, "rel")[0] - roi_means(b, cB, "rel")[0]            # same preference as oddballs
    a, c, r, n = linfit(x, y)
    tuning_vs_oddball[b] = dict(slope=a, intercept=c, r=r, n_roi=n)
    ax.scatter(x, y, color="0.3")
    ax.axhline(0, color="0.5", ls="--", lw=1); ax.axvline(0, color="0.5", ls="--", lw=1)
    if n >= 3:
        xx = np.linspace(np.nanmin(x), np.nanmax(x), 50)
        ax.plot(xx, a * xx + c, color="tab:green", lw=2, label=f"fit: slope = {a:.2f}, r = {r:.2f}")
        ax.legend(frameon=False)
    ax.set_xlabel(f"Control: {oA}° − {oB}° (evoked response)")
    ax.set_ylabel(f"Oddball: {oA}° − {oB}° (relative response)")
    ax.set_title(f"Block {b}: one dot per ROI")
    clean_axes(ax)
fig.tight_layout()
save_fig(fig, "tuning_vs_oddball")
plt.show()
for b, res in tuning_vs_oddball.items():
    print(f"Block {b}: r = {res['r']:.3f}, slope = {res['slope']:.3f} (n = {res['n_roi']} ROIs). "
          "A positive relation means oddball responses follow tuning; no relation means they do not, or that both are too noisy to tell.")

## 15. Distribution of single-trial relative responses

For one ROI, how the relative response varies from trial to trial in each condition.

In [ ]:
def plot_rel_distributions(block, roi=EXAMPLE_ROI, bins=30):
    conds = ["standard", "omission"] + oddball_conditions(block)
    data = {}
    for c in conds:
        m = condition_matrices(block, c)
        data[c] = m["rel"][m["valid_rel"][:, roi], roi]
    print(f"Block {block}, ROI {roi}: " + ", ".join(f"{cond_label(block, c)} n={len(v)}" for c, v in data.items()))

    allv = np.concatenate(list(data.values()))
    lo, hi = np.quantile(allv, [0.01, 0.99])
    pad = 0.1 * (hi - lo + 1e-9)
    xlim = (lo - pad, hi + pad)

    fig, axs = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
    for ax, c in zip(axs.ravel(), conds):
        if len(data[c]):
            ax.hist(data[c], bins=bins, density=True, alpha=0.8, color=cond_color(block, c))
        ax.axvline(0, ls="--", color="0.4")
        ax.set_title(f"{cond_label(block, c)} (n={len(data[c])})")
        ax.set_xlim(*xlim)
        clean_axes(ax)
    for ax in axs[1, :]:
        ax.set_xlabel("Relative response (resp − baseline)")
    for ax in axs[:, 0]:
        ax.set_ylabel("Density")
    fig.suptitle(f"Block {block} | ROI {roi} relative response distributions", y=0.98)
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 5))
    for c in conds:
        if len(data[c]):
            xs = np.sort(data[c])
            ax.plot(xs, np.arange(1, len(xs) + 1) / len(xs), lw=2, color=cond_color(block, c),
                    label=f"{cond_label(block, c)} (n={len(xs)})")
    ax.axvline(0, ls="--", alpha=0.7, color="0.4")
    ax.set_xlim(*xlim)
    ax.set_xlabel("Relative response (resp − baseline)")
    ax.set_ylabel("Cumulative fraction of trials")
    ax.set_title(f"Block {block} | ROI {roi}")
    ax.legend(frameon=False)
    clean_axes(ax)
    plt.show()


plot_rel_distributions(1, roi=N_ROI - 1)

## 16. Standard vs oddball analysis (within each block)

Each point is one ROI. The x-axis is its mean response to the standard; the y-axis is its mean response to an oddball or omission. This is done for the absolute response and for the relative response.

### 16.1 Standard vs each mismatch condition

In [ ]:
def plot_standard_vs_mismatch(block, measure):
    """Three scatter panels (two oddballs, omission) against the standard, with fit and statistics."""
    unit = "mean dF/F" if measure == "abs" else "mean relative dF/F"
    x_std, _ = roi_means(block, "standard", measure)
    conds = mismatch_conditions(block)
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    results = {}
    print(f"\nBlock {block} | {unit} | {N_ROI} ROIs")
    for ax, c in zip(axs, conds):
        y_c, n_c = roi_means(block, c, measure)
        m = np.isfinite(x_std) & np.isfinite(y_c)
        x, y = x_std[m], y_c[m]
        a, b, r, n = linfit(x, y)
        ax.scatter(x, y, color=cond_color(block, c), alpha=0.9, label="ROIs")
        if n >= 2:
            mn, mx = float(min(x.min(), y.min())), float(max(x.max(), y.max()))
            ax.plot([mn, mx], [mn, mx], "k--", label="y = x")
            xx = np.linspace(mn, mx, 100)
            ax.plot(xx, a * xx + b, color=cond_color(block, c), lw=2, label=f"fit: y = {a:.2f}x + {b:.3f}")
            ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(f"{cond_label(block, 'standard')} ({unit})")
        ax.set_ylabel(f"{cond_label(block, c)} ({unit})")
        ax.set_title(f"Block {block}: standard vs {cond_label(block, c)}")
        ax.legend(frameon=False)
        clean_axes(ax)

        delta = y - x
        results[c] = dict(slope=a, intercept=b, r=r, n_roi=n,
                          delta_mean=float(np.mean(delta)), delta_sd=float(np.std(delta)))
        print(f"\n--- {cond_label(block, 'standard')} vs {cond_label(block, c)} "
              f"(kept trials, median over ROIs: {int(np.median(n_c))}) ---")
        print(f"Valid ROIs used: {n} / {N_ROI}")
        print(f"Standard mean ± SD : {np.mean(x):.4f} ± {np.std(x):.4f}")
        print(f"{cond_label(block, c)} mean ± SD : {np.mean(y):.4f} ± {np.std(y):.4f}")
        print(f"Δ (condition − standard) mean ± SD: {np.mean(delta):+.4f} ± {np.std(delta):.4f}")
        print(f"% ROIs with condition > standard: {(delta > 0).mean() * 100:.1f}%")
        print(f"Fit: slope = {a:.4f}, intercept = {b:.4f}, Pearson r = {r:.4f}")
    fig.tight_layout()
    save_fig(fig, f"standard_vs_mismatch_{measure}_b{block}")
    plt.show()
    return results


standard_vs_mismatch = {(b, m): plot_standard_vs_mismatch(b, m) for b in BLOCKS for m in ["abs", "rel"]}

### 16.2 The two oddballs on one plot, with fit intersection and area (Fig. 4)

Both oddball fits share the same x-axis (the standard response). The plot marks where the two fitted lines cross and shades the area between them from the crossing point to the largest standard response. The signed area is the first oddball's line minus the 45° line.

Optional trimming removes extreme ROIs before the final fit:

- `trim="residual"`: removes ROIs whose absolute residual is above the 98th percentile
- `trim="influence"`: removes ROIs flagged by Cook's distance, leverage or studentized residual

An ROI is removed from **both** fits if it fails **either** one.

In [ ]:
def ols_influence(x, y):
    """
    OLS y = a*x + b with influence diagnostics.
    Returns: a, b, se_a, se_b, r, resid, h (leverage), rstudent, cooks
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    n = x.size
    p = 2
    if n < 3:
        return None

    X = np.column_stack([x, np.ones(n)])
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    a, b = float(beta[0]), float(beta[1])
    resid = y - X @ beta

    s2 = (resid @ resid) / max(n - p, 1)
    cov_beta = s2 * XtX_inv
    se_a = float(np.sqrt(cov_beta[0, 0]))
    se_b = float(np.sqrt(cov_beta[1, 1]))

    h = np.clip(np.diag(X @ XtX_inv @ X.T), 0, 1)
    r_int = resid / np.sqrt(np.maximum(s2 * (1 - h), 1e-12))
    rstudent = r_int * np.sqrt(np.maximum((n - p - 1) / np.maximum((n - p) - r_int**2, 1e-12), 0))
    cooks = (r_int**2) * (h / (p * np.maximum(1 - h, 1e-12)))
    r = float(np.corrcoef(x, y)[0, 1]) if n >= 2 else np.nan
    return dict(n=n, a=a, b=b, se_a=se_a, se_b=se_b, r=r, resid=resid, h=h, rstudent=rstudent, cooks=cooks)


def influence_keep(x, y, rstud_rule=3.0):
    """ROIs that pass Cook's distance (4/n), leverage (2p/n) and studentized residual thresholds for one fit."""
    d = ols_influence(x, y)
    n, p = x.size, 2
    return (d["cooks"] <= 4.0 / n) & (d["h"] <= 2 * p / n) & (np.abs(d["rstudent"]) <= rstud_rule)


def plot_oddball_wedge(block, measure, trim=None, trim_q=0.98, rstud_rule=3.0, show=True):
    """
    Standard (x) vs the two oddballs (y) for one block.
    measure: "abs" or "rel".   trim: None, "residual" or "influence".
    Returns a dict with the fits, removed ROIs, intersection and signed area.
    """
    unit = "mean dF/F" if measure == "abs" else "mean relative ΔF/F"
    cA, cB = oddball_conditions(block)                     # A = far oddball (blue), B = 45° (red)
    x_all, _ = roi_means(block, "standard", measure)
    yA_all, _ = roi_means(block, cA, measure)
    yB_all, _ = roi_means(block, cB, measure)

    roi_idx = np.flatnonzero(np.isfinite(x_all) & np.isfinite(yA_all) & np.isfinite(yB_all))
    x_all, yA_all, yB_all = x_all[roi_idx], yA_all[roi_idx], yB_all[roi_idx]
    n = x_all.size

    # ---------- trimming ----------
    keepA = np.ones(n, dtype=bool)
    keepB = np.ones(n, dtype=bool)
    if trim is not None and n >= 5:
        if trim == "residual":
            for y_all, keep in ((yA_all, keepA), (yB_all, keepB)):
                a0, b0 = np.polyfit(x_all, y_all, 1)
                res = np.abs(y_all - (a0 * x_all + b0))
                keep[:] = res <= np.quantile(res, trim_q)
        elif trim == "influence":
            keepA = influence_keep(x_all, yA_all, rstud_rule)
            keepB = influence_keep(x_all, yB_all, rstud_rule)
        else:
            raise ValueError("trim must be None, 'residual' or 'influence'")
    keep = keepA & keepB
    if keep.sum() < 3:
        keep[:] = keepA[:] = keepB[:] = True
    x, yA, yB = x_all[keep], yA_all[keep], yB_all[keep]

    # ---------- final fits ----------
    fitA, fitB = ols_influence(x, yA), ols_influence(x, yB)
    aA, bA, aB, bB = fitA["a"], fitA["b"], fitB["a"], fitB["b"]

    # ---------- intersection and signed area (A − B) from the crossing to max(x) ----------
    x_cross = y_cross = None
    area = np.nan
    if abs(aA - aB) > 1e-12:
        x_cross = (bB - bA) / (aA - aB)
        y_cross = aA * x_cross + bA
        x_end = float(x.max())
        if x_end > x_cross:
            area = 0.5 * (aA - aB) * (x_end**2 - x_cross**2) + (bA - bB) * (x_end - x_cross)

    out = {
        "block": block, "measure": measure, "trim": trim or "none",
        "n_roi_raw": int(n), "n_roi_used": int(keep.sum()),
        "removed_rois": roi_idx[~keep].tolist(),
        "removed_by_first": roi_idx[~keepA].tolist(), "removed_by_second": roi_idx[~keepB].tolist(),
        "fits": {
            cA: dict(slope=aA, intercept=bA, se_slope=fitA["se_a"], se_intercept=fitA["se_b"], r=fitA["r"]),
            cB: dict(slope=aB, intercept=bB, se_slope=fitB["se_a"], se_intercept=fitB["se_b"], r=fitB["r"]),
        },
        "x_cross": None if x_cross is None else float(x_cross),
        "y_cross": None if y_cross is None else float(y_cross),
        "area_signed": None if not np.isfinite(area) else float(area),
    }

    # ---------- plot ----------
    if show:
        colA, colB = cond_color(block, cA), cond_color(block, cB)
        if trim is None:
            lo = np.nanmin([x.min(), yA.min(), yB.min()])
            hi = np.nanmax([x.max(), yA.max(), yB.max()])
            margin = 0.05 * (hi - lo)
            lim = (lo - margin, hi + margin)
        else:
            xq = np.quantile(x, [0.01, 0.99])
            yq = np.quantile(np.r_[yA, yB], [0.01, 0.99])
            lim = (min(xq[0], yq[0]), max(xq[1], yq[1]))

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.scatter(x, yA, color=colA, alpha=0.9, label=cond_label(block, cA))
        ax.scatter(x, yB, color=colB, alpha=0.9, label=cond_label(block, cB))
        if trim is not None and (~keep).any():
            ax.scatter(x_all[~keepA], yA_all[~keepA], facecolors="none", edgecolors=colA, s=70, lw=1.2,
                       label=f"removed by {cond_label(block, cA)} fit")
            ax.scatter(x_all[~keepB], yB_all[~keepB], facecolors="none", edgecolors=colB, s=70, lw=1.2,
                       label=f"removed by {cond_label(block, cB)} fit")
        ax.plot(lim, lim, "k--", label="y = x")
        xx = np.linspace(lim[0], lim[1], 200)
        ax.plot(xx, aA * xx + bA, color=colA, label=f"fit: y = {aA:.2f}x + {bA:.3f}")
        ax.plot(xx, aB * xx + bB, color=colB, label=f"fit: y = {aB:.2f}x + {bB:.3f}")
        if x_cross is not None:
            ax.scatter([x_cross], [y_cross], color="k", marker="x", zorder=5, label="fit intersection")
            if np.isfinite(area):
                xf = np.linspace(x_cross, float(x.max()), 200)
                ax.fill_between(xf, aA * xf + bA, aB * xf + bB, where=(aA * xf + bA > aB * xf + bB),
                                color="orange", alpha=0.2, label="area between fits")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(f"{cond_label(block, 'standard')} ({unit})")
        ax.set_ylabel(f"Oddball ({unit})")
        trim_txt = "" if trim is None else f", {trim} trim"
        ax.set_title(f"{DMD} Block {block} | {'absolute' if measure == 'abs' else 'relative'} response{trim_txt}\n"
                     f"(n ROI = {out['n_roi_used']} of {n})")
        ax.legend(frameon=False, fontsize=9)
        clean_axes(ax)
        save_fig(fig, f"oddball_wedge_{measure}_{trim or 'notrim'}_b{block}")
        plt.show()

    print(f"[Block {block} | {measure} | trim = {trim or 'none'}] n used = {out['n_roi_used']} / {n}"
          + (f", removed ROIs {out['removed_rois']}" if out["removed_rois"] else ""))
    for c in (cA, cB):
        f = out["fits"][c]
        print(f"  {cond_label(block, c)}: slope = {f['slope']:.4f} ± {f['se_slope']:.4f}, "
              f"intercept = {f['intercept']:.4f} ± {f['se_intercept']:.4f}, r = {f['r']:.4f}")
    if x_cross is None:
        print("  Fits are parallel: no intersection.")
    elif out["area_signed"] is None:
        print(f"  Intersection x* = {x_cross:.4f} is to the right of all data: no area over the data range.")
    else:
        print(f"  Intersection x* = {x_cross:.4f}, y* = {y_cross:.4f}; signed area = {area:.6f}")
    return out


wedge_results = {}
for _b in BLOCKS:
    wedge_results[(_b, "abs", "none")] = plot_oddball_wedge(_b, "abs")
    wedge_results[(_b, "rel", "none")] = plot_oddball_wedge(_b, "rel")

### 16.3 Relative response with trimming (Fig. 5c–d)

In [ ]:
for _b in BLOCKS:
    wedge_results[(_b, "rel", "influence")] = plot_oddball_wedge(_b, "rel", trim="influence")
    wedge_results[(_b, "rel", "residual")]  = plot_oddball_wedge(_b, "rel", trim="residual")

### 16.4 How far can the slopes be trusted? (slope checks)

A fitted slope is pulled towards zero when the x-axis values are noisy, and it can change a lot when a few ROIs move. This table reports, for every standard-vs-oddball fit:

- **slope and intercept** with a 95% bootstrap interval (ROIs resampled 2,000 times)
- **through-origin slope:** the fit forced through (0, 0)
- **Deming slope:** a fit that allows for noise on both axes
- **reliability of x and y:** how reproducible the per-ROI values are when the trials are split in half (1 = perfectly reproducible, 0 = noise)
- **corrected slope:** the slope divided by the reliability of x, which undoes the pull towards zero

In [ ]:
N_BOOT = 2000


def split_half_reliability(block, condition, measure, n_splits=200, seed=RANDOM_SEED):
    """Reproducibility of the per-ROI means across random halves of the trials (Spearman-Brown corrected)."""
    col, valid = MEASURES[measure]
    m = condition_matrices(block, condition)
    vals = np.where(m[valid], m[col], np.nan)
    vals = vals[np.isfinite(vals).any(axis=1)]
    n = vals.shape[0]
    if n < 4:
        return np.nan
    rng = np.random.default_rng(seed)
    rs = []
    for _ in range(n_splits):
        perm = rng.permutation(n)
        r = pearson(nanmean_quiet(vals[perm[: n // 2]], axis=0), nanmean_quiet(vals[perm[n // 2:]], axis=0))
        if np.isfinite(r):
            rs.append(2 * r / (1 + r) if r > -1 else np.nan)
    return float(np.nanmedian(rs)) if rs else np.nan


def error_variance(block, condition, measure):
    """Mean over ROIs of the squared standard error of the per-ROI mean."""
    col, valid = MEASURES[measure]
    m = condition_matrices(block, condition)
    vals = np.where(m[valid], m[col], np.nan)
    n = np.isfinite(vals).sum(axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return float(np.nanmean(np.nanvar(vals, axis=0, ddof=1) / n))


def slope_checks(x, y, err_var_x, err_var_y, rel_x, rel_y, n_boot=N_BOOT, seed=RANDOM_SEED):
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    a, b, r, n = linfit(x, y)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if np.ptp(x[idx]) > 0:
            boots.append(np.polyfit(x[idx], y[idx], 1))
    boots = np.array(boots)
    sxx, syy, sxy = np.var(x, ddof=1), np.var(y, ddof=1), np.cov(x, y)[0, 1]
    delta = err_var_y / err_var_x if err_var_x > 0 else np.nan
    # Deming fit: only meaningful when x and y are clearly related
    deming = ((syy - delta * sxx + np.sqrt((syy - delta * sxx) ** 2 + 4 * delta * sxy ** 2)) / (2 * sxy)
              if np.isfinite(delta) and abs(r) >= 0.3 else np.nan)
    return {
        "n ROI": n, "slope": a,
        "slope 2.5%": np.percentile(boots[:, 0], 2.5), "slope 97.5%": np.percentile(boots[:, 0], 97.5),
        "intercept": b,
        "intercept 2.5%": np.percentile(boots[:, 1], 2.5), "intercept 97.5%": np.percentile(boots[:, 1], 97.5),
        "through-origin slope": float(np.sum(x * y) / np.sum(x * x)),
        "Deming slope": float(deming),
        "reliability of x": rel_x, "reliability of y": rel_y,
        "corrected slope": a / rel_x if np.isfinite(rel_x) and rel_x > 0.2 else np.nan,
    }


def slope_check_row(label, bx, cx, by, cy, measure):
    x, _ = roi_means(bx, cx, measure)
    y, _ = roi_means(by, cy, measure)
    out = {"comparison": label, "measure": measure}
    out.update(slope_checks(x, y, error_variance(bx, cx, measure), error_variance(by, cy, measure),
                            split_half_reliability(bx, cx, measure), split_half_reliability(by, cy, measure)))
    return out


_rows = []
for _b in BLOCKS:
    for _m in ["abs", "rel"]:
        for _c in oddball_conditions(_b):
            _rows.append(slope_check_row(f"B{_b}: {cond_label(_b, 'standard')} vs {cond_label(_b, _c)}",
                                         _b, "standard", _b, _c, _m))
slope_check_table = pd.DataFrame(_rows)
display(slope_check_table.round(3))
print("Reliability below about 0.5 means the per-ROI values are mostly noise, and a slope fitted to them says little.\n"
      "The corrected slope is left empty when the reliability of x is below 0.2, "
      "and the Deming slope when x and y are barely related (|r| < 0.3).")

### 16.5 Is the absolute response just each ROI's overall level? (per-ROI normalisation)

In the absolute scatter plots every ROI sits close to the unity line. One explanation is tuning that is preserved across conditions. Another is simpler: each ROI has its own overall dF/F level, and that level shows up in every condition.

- **Left:** absolute response to the standard against the ROI's mean dF/F over the whole block (all samples, stimulus or not).
- **Right:** the oddball effect after removing each ROI's own level and noise: `z = (oddball − standard) / (SD of the standard's trials / √n)`. Values within ±1.96 are what the trial-to-trial noise of that ROI gives by chance.

In [ ]:
def normalised_effects(block):
    rows = []
    m_std = condition_matrices(block, "standard")
    std_vals = np.where(m_std["valid_abs"], m_std["resp"], np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        sd_trial = np.nanstd(std_vals, axis=0, ddof=1)
    A_std, _ = roi_means(block, "standard", "abs")
    level = nanmean_quiet(BLOCKS[block]["dff"], axis=0)                 # ROI's mean dF/F over the whole block
    for c in mismatch_conditions(block):
        A_c, n_c = roi_means(block, c, "abs")
        z = (A_c - A_std) / (sd_trial / np.sqrt(np.maximum(n_c, 1)))
        mi = (A_c - A_std) / (np.abs(A_c) + np.abs(A_std))
        for r in range(N_ROI):
            rows.append({"block": block, "condition": c, "roi": r, "block level": level[r], "standard": A_std[r],
                         "condition response": A_c[r], "trial SD": sd_trial[r], "z": z[r], "modulation index": mi[r]})
    return pd.DataFrame(rows)


normalised = pd.concat([normalised_effects(b) for b in BLOCKS], ignore_index=True)

fig, axs = plt.subplots(len(BLOCKS), 2, figsize=(12, 4.6 * len(BLOCKS)), squeeze=False)
_rows = []
for i, b in enumerate(BLOCKS):
    sub = normalised[normalised["block"] == b]
    one = sub[sub["condition"] == mismatch_conditions(b)[0]]
    a, c0, r, n = linfit(one["block level"], one["standard"])
    ax = axs[i, 0]
    ax.scatter(one["block level"], one["standard"], color="k")
    lim = (min(one["block level"].min(), one["standard"].min()), max(one["block level"].max(), one["standard"].max()))
    ax.plot(lim, lim, "k--", lw=1, label="y = x")
    ax.set_xlabel("Mean dF/F over the whole block")
    ax.set_ylabel(f"Response to {cond_label(b, 'standard')}")
    ax.set_title(f"Block {b}: r = {r:.3f}, slope = {a:.2f}")
    ax.legend(frameon=False)
    clean_axes(ax)

    ax = axs[i, 1]
    rng = np.random.default_rng(RANDOM_SEED)
    for j, c in enumerate(mismatch_conditions(b)):
        z = sub.loc[sub["condition"] == c, "z"].to_numpy()
        ax.scatter(j + rng.uniform(-0.12, 0.12, z.size), z, color=cond_color(b, c), s=30)
        ax.errorbar(j + 0.28, np.nanmean(z), yerr=sem(z), fmt="o", color="k", capsize=4)
        _rows.append({"block": b, "condition": cond_label(b, c), "mean z": np.nanmean(z),
                      "ROIs with |z| > 1.96": int(np.sum(np.abs(z) > 1.96)), "ROIs": int(np.isfinite(z).sum()),
                      "mean modulation index": float(sub.loc[sub["condition"] == c, "modulation index"].mean()),
                      "level vs standard r": r})
    for y0 in (-1.96, 1.96):
        ax.axhline(y0, color="0.5", ls="--", lw=1)
    ax.axhline(0, color="0.5", lw=0.8)
    ax.set_xticks(range(3))
    ax.set_xticklabels([cond_label(b, c) for c in mismatch_conditions(b)])
    ax.set_ylabel("z (condition − standard)")
    ax.set_title(f"Block {b}: one dot per ROI (black = mean ± SEM)")
    clean_axes(ax)
fig.tight_layout()
save_fig(fig, "normalised_effects")
plt.show()
normalised_summary = pd.DataFrame(_rows)
display(normalised_summary.round(3))

### 16.6 Pooling the ROIs: is there a mismatch effect at the level of the whole imaging plane?

A single ROI with 30 trials is too noisy to show a small effect. Averaging over the ROIs gives one number per condition: the mean over ROIs of (condition − standard).

The chance range comes from standard samples with the same number of trials (section 11). Drawing whole trials keeps the noise that ROIs share on a trial, so the ROIs are not treated as independent.

In [ ]:
def population_test(block, measure):
    S, _ = roi_means(block, "standard", measure)
    rows = []
    for c in mismatch_conditions(block):
        V, n = roi_means(block, c, measure)
        real = float(np.nanmean(V - S))
        chance = np.nanmean(standard_samples(block, measure, int(np.median(n))) - S, axis=1)
        lo, hi = np.percentile(chance, [2.5, 97.5])
        rows.append({"block": block, "measure": measure, "condition": c, "trials": int(np.median(n)),
                     "effect (mean over ROIs)": real, "chance 2.5%": lo, "chance 97.5%": hi,
                     "effect in chance SDs": (real - np.mean(chance)) / np.std(chance), "p": chance_p(real, chance)})
    return pd.DataFrame(rows)


population_table = pd.concat([population_test(b, m) for b in BLOCKS for m in ["abs", "rel"]], ignore_index=True)

fig, axs = plt.subplots(1, 2 * len(BLOCKS), figsize=(4.6 * 2 * len(BLOCKS), 4.2), squeeze=False)
for ax, (b, m) in zip(axs[0], [(b, m) for b in BLOCKS for m in ["abs", "rel"]]):
    sub = population_table[(population_table["block"] == b) & (population_table["measure"] == m)].reset_index(drop=True)
    for j, row in sub.iterrows():
        ax.fill_between([j - 0.3, j + 0.3], row["chance 2.5%"], row["chance 97.5%"], color="0.8")
        ax.scatter(j, row["effect (mean over ROIs)"], color=cond_color(b, row["condition"]), s=70, zorder=5)
        ax.text(j, row["chance 97.5%"], f"p={row['p']:.3f}", ha="center", va="bottom", fontsize=10)
    ax.axhline(0, color="0.5", ls="--", lw=1)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([cond_label(b, c).replace("Oddball ", "Odd ") for c in sub["condition"]])
    ax.set_title(f"Block {b}, {'absolute' if m == 'abs' else 'relative'}")
    ax.set_ylabel("Condition − standard")
    clean_axes(ax)
fig.suptitle("Dot: real effect. Grey: 95% range of standard samples with the same trial count.", y=1.03)
fig.tight_layout()
save_fig(fig, "population_test")
plt.show()
_show = population_table.copy()
_show["condition"] = [cond_label(b, c) for b, c in zip(_show["block"], _show["condition"])]
display(_show.round(4))

## 17. Around the oddball: before, during and after

During the baseline the mouse cannot know what comes next, so the informative moments are the mismatch trial itself and what follows it.

For every mismatch trial that has a standard directly before and directly after it, three trials are lined up: **standard before → mismatch → standard after**. Runs of two or more mismatch trials are left out here.

**Chance reference:** three standards in a row, drawn at random in the same number.

### 17.1 Time course across the three trials (mean of all ROIs)

In [ ]:
def triplet_anchors(block):
    """Start times of the middle trial of clean triplets.
    Returns {condition: starts} for mismatch trials, and the starts of standards that sit between two standards."""
    tr = TRIPLETS[block]
    bt = (trial_table[(trial_table["block"] == block) & (trial_table["roi"] == 0)]
          .sort_values("trial_in_block").reset_index(drop=True))
    start_of = dict(zip(bt["trial_in_block"], bt["start_time"]))
    clean = tr[(tr["run"].apply(len) == 1) & tr["pre"].notna() & tr["post"].notna()]
    anchors = {c: np.array([start_of[k] for k in g["trial"]]) for c, g in clean.groupby("condition")}
    role = bt["role"].to_numpy()
    ok = np.zeros(len(bt), bool)
    ok[2:-1] = ((role[2:-1] == "standard") & (role[1:-2] == "standard") & (role[3:] == "standard")
                & (role[:-3] == "standard"))
    ok &= bt["in_block"].to_numpy()
    return anchors, bt.loc[ok, "start_time"].to_numpy()


def population_traces(block, starts, t_before, t_after):
    """Mean-of-all-ROIs trace around each start time, on a common time axis."""
    t, dff = BLOCKS[block]["t"], BLOCKS[block]["dff"]
    pop = nanmean_quiet(dff, axis=1)
    common_t = np.arange(-t_before, t_after, np.median(np.diff(t)))
    traces = []
    for t0 in starts:
        i0 = np.searchsorted(t, t0 - t_before, side="left")
        i1 = np.searchsorted(t, t0 + t_after, side="right")
        x, y = t[i0:i1] - t0, pop[i0:i1]
        ok = np.isfinite(y)
        if x.size < 3 or ok.mean() < 0.5 or x[ok].min() > -t_before + 0.1 or x[ok].max() < t_after - 0.1:
            continue
        traces.append(np.interp(common_t, x[ok], y[ok]))
    return common_t, (np.vstack(traces) if traces else np.empty((0, common_t.size)))


def plot_around_oddball(block):
    anchors, std_anchors = triplet_anchors(block)
    stim = BLOCKS[block]["stim"].sort_values("start_time")
    soa = float(np.median(np.diff(stim["start_time"])))                       # onset-to-onset time
    dur = float(np.median(stim["stop_time"] - stim["start_time"]))
    t_before, t_after = soa + PRE_WIN, soa + dur + POST_WIN
    common_t, S = population_traces(block, std_anchors, t_before, t_after)

    conds = [c for c in mismatch_conditions(block) if c in anchors]
    fig, axs = plt.subplots(len(conds), 1, figsize=(12, 3.2 * len(conds)), sharex=True, squeeze=False)
    rng = np.random.default_rng(RANDOM_SEED)
    for ax, c in zip(axs[:, 0], conds):
        _, C = population_traces(block, anchors[c], t_before, t_after)
        n = len(C)
        draws = np.array([S[rng.choice(len(S), min(n, len(S)), replace=False)].mean(axis=0) for _ in range(N_DRAWS)])
        lo, hi = np.percentile(draws, [2.5, 97.5], axis=0)
        ax.fill_between(common_t, lo, hi, color="0.6", alpha=0.35, lw=0, label=f"95% range of {n} standard triplets")
        ax.plot(common_t, S.mean(axis=0), color="k", lw=1.6, label=f"Three standards in a row (n={len(S)})")
        ax.plot(common_t, C.mean(axis=0), color=cond_color(block, c), lw=1.8,
                label=f"Standard → {cond_label(block, c)} → standard (n={n})")
        for k, name in [(-1, "standard before"), (0, cond_label(block, c)), (1, "standard after")]:
            ax.axvspan(k * soa, k * soa + dur, color=cond_color(block, c) if k == 0 else "0.5", alpha=0.10)
            ax.text(k * soa + dur / 2, 1.0, name, transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=10)
        ax.set_ylabel("dF/F (mean of all ROIs)")
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=10)
        clean_axes(ax)
    axs[-1, 0].set_xlabel("Time relative to the onset of the middle trial (s)")
    fig.suptitle(f"Block {block}", y=1.02)
    fig.tight_layout()
    save_fig(fig, f"around_oddball_b{block}")
    plt.show()


# TRIPLETS comes from section 12.3
for _b in BLOCKS:
    plot_around_oddball(_b)

### 17.2 Three numbers per mismatch condition

All values are the absolute response or baseline, averaged over the ROIs.

- **mismatch − standard before:** the mismatch response against its own matched control trial.
- **baseline after − baseline before:** the grey screen right after the mismatch trial against the grey screen before the standard that preceded it. This is the "shift everything by one window" question.
- **standard after − standard before:** the response to the first standard after the mismatch against the standard before it.

Chance: the same three differences for random sets of three standards in a row.

In [ ]:
def triplet_values(block):
    bt = trial_table[trial_table["block"] == block]
    resp = bt.pivot(index="trial_in_block", columns="roi", values="resp").where(
        bt.pivot(index="trial_in_block", columns="roi", values="valid_abs"))
    base = bt.pivot(index="trial_in_block", columns="roi", values="base").where(
        bt.pivot(index="trial_in_block", columns="roi", values="valid_rel"))
    resp_pop, base_pop = resp.mean(axis=1), base.mean(axis=1)              # mean over ROIs, per trial

    def differences(middle_trials):
        k = np.asarray(middle_trials)
        d = pd.DataFrame({
            "mismatch − standard before": resp_pop.reindex(k).to_numpy() - resp_pop.reindex(k - 1).to_numpy(),
            "baseline after − baseline before": base_pop.reindex(k + 1).to_numpy() - base_pop.reindex(k - 1).to_numpy(),
            "standard after − standard before": resp_pop.reindex(k + 1).to_numpy() - resp_pop.reindex(k - 1).to_numpy(),
        })
        return d

    tr = TRIPLETS[block]
    clean = tr[(tr["run"].apply(len) == 1) & tr["pre"].notna() & tr["post"].notna()]
    one = bt[bt["roi"] == 0].sort_values("trial_in_block")
    role = one["role"].to_numpy()
    k_all = one["trial_in_block"].to_numpy()
    ok = np.zeros(len(one), bool)
    ok[2:-1] = (role[2:-1] == "standard") & (role[1:-2] == "standard") & (role[3:] == "standard") & (role[:-3] == "standard")
    chance_all = differences(k_all[ok & one["in_block"].to_numpy()])

    rng = np.random.default_rng(RANDOM_SEED)
    rows = []
    for c in mismatch_conditions(block):
        real = differences(clean.loc[clean["condition"] == c, "trial"].to_numpy())
        for name in real.columns:
            v = real[name].dropna().to_numpy()
            pool = chance_all[name].dropna().to_numpy()
            if v.size < 3 or pool.size < v.size:
                continue
            chance = np.array([pool[rng.choice(pool.size, v.size, replace=False)].mean() for _ in range(N_DRAWS)])
            lo, hi = np.percentile(chance, [2.5, 97.5])
            rows.append({"block": block, "condition": cond_label(block, c), "difference": name, "triplets": v.size,
                         "mean": v.mean(), "chance 2.5%": lo, "chance 97.5%": hi, "p": chance_p(v.mean(), chance)})
    return pd.DataFrame(rows)


around_table = pd.concat([triplet_values(b) for b in BLOCKS], ignore_index=True)
display(around_table.round(4))
print(f"{int((around_table['p'] < 0.05).sum())} of {len(around_table)} comparisons are outside the 95% chance range; "
      f"about {0.05 * len(around_table):.0f} would be expected by chance alone.")

## 18. Baseline analysis (within each block)

Does the pre-stimulus baseline differ between standard trials and mismatch trials, and is that difference related to the relative response?

Notation, per ROI:

- `B` = mean baseline, `Rel` = mean relative response
- `Δbaseline = B(mismatch) − B(standard)`
- `Δrelative = Rel(mismatch) − Rel(standard)`

### 18.1 Baseline on standard trials vs baseline on mismatch trials (Fig. 6a–c)

In [ ]:
def plot_baseline_scatter(block):
    B_std, n_std = roi_means(block, "standard", "base")
    conds = mismatch_conditions(block)
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    results = {}
    print(f"\nBlock {block} ROI baselines (mean dF/F in [start − {PRE_WIN} s, start))")
    print(f"Kept standard trials per ROI (min/median/max): {n_std.min()} / {np.median(n_std)} / {n_std.max()}")
    for ax, c in zip(axs, conds):
        B_c, n_c = roi_means(block, c, "base")
        m = np.isfinite(B_std) & np.isfinite(B_c)
        x, y = B_std[m], B_c[m]
        a, b, r, n = linfit(x, y)
        col = cond_color(block, c)
        ax.scatter(x, y, color=col, alpha=0.9, label="ROIs")
        if n >= 2:
            lo, hi = float(min(x.min(), y.min())), float(max(x.max(), y.max()))
            margin = 0.1 * (hi - lo) if hi > lo else 0.01
            lim = (lo - margin, hi + margin)
            ax.plot(lim, lim, "k--", label="y = x")
            xx = np.linspace(lim[0], lim[1], 100)
            ax.plot(xx, a * xx + b, color=col, lw=2, label=f"fit: y = {a:.2f}x + {b:.3f}")
            ax.set_xlim(lim); ax.set_ylim(lim)
            ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("Standard baseline (absolute dF/F)")
        ax.set_ylabel(f"{cond_label(block, c)} baseline (absolute dF/F)")
        ax.legend(frameon=False)
        clean_axes(ax)

        delta = y - x
        results[c] = dict(slope=a, intercept=b, r=r, n_roi=n,
                          delta_mean=float(delta.mean()), delta_sd=float(delta.std()))
        print(f"\n--- Standard baseline vs {cond_label(block, c)} baseline "
              f"(kept trials, median over ROIs: {int(np.median(n_c))}) ---")
        print(f"Standard baseline mean ± SD : {x.mean():.4f} ± {x.std():.4f}")
        print(f"{cond_label(block, c)} baseline mean ± SD : {y.mean():.4f} ± {y.std():.4f}")
        print(f"Δbaseline mean ± SD: {delta.mean():+.4f} ± {delta.std():.4f}")
        print(f"% ROIs with higher baseline than standard: {(delta > 0).mean() * 100:.1f}%")
        print(f"Fit: slope = {a:.4f}, intercept = {b:.4f}, r = {r:.4f}")
    fig.tight_layout()
    save_fig(fig, f"baseline_scatter_b{block}")
    plt.show()
    return results


baseline_scatter = {b: plot_baseline_scatter(b) for b in BLOCKS}

### 18.2 Does the baseline shift depend on the starting baseline level?

In [ ]:
def plot_baseline_shift(block):
    B_std, _ = roi_means(block, "standard", "base")
    fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, c in zip(axs, mismatch_conditions(block)):
        B_c, _ = roi_means(block, c, "base")
        ax.scatter(B_std, B_c - B_std, color=cond_color(block, c))
        ax.axhline(0, color="0.5", ls="--", lw=1)
        ax.set_xlabel("Standard baseline (absolute dF/F)")
        ax.set_title(f"Block {block}: {cond_label(block, c)}")
        clean_axes(ax)
    axs[0].set_ylabel("Δbaseline (absolute dF/F)")
    fig.tight_layout()
    save_fig(fig, f"baseline_shift_b{block}")
    plt.show()


for _b in BLOCKS:
    plot_baseline_shift(_b)

### 18.3 Baseline shift vs relative mismatch response, and state dependence (Fig. 6d–i)

Left column: `Δbaseline` against `Δrelative`. Right column: standard baseline against `Δrelative`.

**Caution.** The relative response is the absolute response minus the baseline, so `Δrelative` already contains `−Δbaseline`. A negative correlation in the left column is therefore expected even from noise alone. Section 19 tests how much of it is more than that.

In [ ]:
def baseline_metrics(block, condition):
    """corr(Δbaseline, Δrelative) and the state-dependence fit (Δrelative against standard baseline)."""
    B_std, _ = roi_means(block, "standard", "base")
    R_std, _ = roi_means(block, "standard", "rel")
    B_c, _ = roi_means(block, condition, "base")
    R_c, _ = roi_means(block, condition, "rel")
    dB, dRel = B_c - B_std, R_c - R_std
    m = np.isfinite(dB) & np.isfinite(dRel) & np.isfinite(B_std)
    out = dict(n_roi=int(m.sum()), corr_dB_dRel=np.nan, state_slope=np.nan, state_corr=np.nan,
               dB=dB, dRel=dRel, B_std=B_std)
    if m.sum() >= 3:
        out["corr_dB_dRel"] = float(np.corrcoef(dB[m], dRel[m])[0, 1])
        a, b = np.polyfit(B_std[m], dRel[m], 1)
        out["state_slope"] = float(a)
        out["state_corr"] = float(np.corrcoef(B_std[m], dRel[m])[0, 1])
    return out


def plot_baseline_vs_relative(block):
    conds = mismatch_conditions(block)
    metrics = {c: baseline_metrics(block, c) for c in conds}

    vals = np.concatenate([np.r_[metrics[c]["dB"], metrics[c]["dRel"]] for c in conds])
    vals = vals[np.isfinite(vals)]
    lim_val = 1.05 * np.max(np.abs(vals)) if vals.size else 1.0
    lims = (-lim_val, lim_val)
    ticks = [-0.1, 0.0, 0.1]

    fig, axs = plt.subplots(3, 2, figsize=(11, 12))
    for (ax_l, ax_r), c in zip(axs, conds):
        mt, col, lab = metrics[c], cond_color(block, c), cond_label(block, c)
        m = np.isfinite(mt["dB"]) & np.isfinite(mt["dRel"])
        ax_l.scatter(mt["dB"][m], mt["dRel"][m], color=col, s=40)
        ax_l.axhline(0, color="0.5", ls="--", lw=1)
        ax_l.axvline(0, color="0.5", ls="--", lw=1)
        ax_l.set_xlim(lims); ax_l.set_ylim(lims)
        ax_l.set_xticks(ticks); ax_l.set_yticks(ticks)
        ax_l.set_aspect("equal", adjustable="box")
        ax_l.set_xlabel(f"Δbaseline ({lab})")
        ax_l.set_ylabel(f"Δrelative ({lab})")
        ax_l.text(0.04, 0.90, f"n = {mt['n_roi']}\nr = {mt['corr_dB_dRel']:.3f}",
                  transform=ax_l.transAxes, ha="left", va="top", fontsize=13)

        m2 = np.isfinite(mt["B_std"]) & np.isfinite(mt["dRel"])
        ax_r.scatter(mt["B_std"][m2], mt["dRel"][m2], color=col, s=40)
        ax_r.axhline(0, color="0.5", ls="--", lw=1)
        ax_r.set_ylim(lims); ax_r.set_yticks(ticks)
        ax_r.set_xlabel("Standard baseline (absolute dF/F)")
        ax_r.set_ylabel(f"Δrelative ({lab})")
        ax_r.text(0.04, 0.90, f"r = {mt['state_corr']:.3f}\nslope = {mt['state_slope']:.3f}",
                  transform=ax_r.transAxes, ha="left", va="top", fontsize=13)
        clean_axes(ax_l); clean_axes(ax_r)
    fig.suptitle(f"Block {block}", y=0.995)
    fig.tight_layout()
    save_fig(fig, f"baseline_vs_relative_b{block}")
    plt.show()

    for c in conds:
        mt = metrics[c]
        print(f"Block {block} | {cond_label(block, c):12s}: n ROI = {mt['n_roi']}, "
              f"corr(Δbaseline, Δrelative) = {mt['corr_dB_dRel']:.3f}, "
              f"state corr = {mt['state_corr']:.3f}, state slope = {mt['state_slope']:.4f}")
    return metrics


baseline_results = {b: plot_baseline_vs_relative(b) for b in BLOCKS}

## 19. Is the baseline result more than a built-in effect?

Four checks, following the feedback from the OpenScope meeting.

### 19.1 Fake oddballs: what does chance look like?

On oddball trials the baseline comes from only about 28 trials, so `Δbaseline` is noisy, and that noise enters `Δrelative` with the opposite sign.

To see what this produces by itself, the same number of **standard** trials is drawn at random and treated as if they were an oddball condition. Nothing differs between these trials and the other standards, so any correlation they give is the built-in one. This is repeated 1,000 times and compared with the real value.

- Only standards that follow another standard are drawn, because oddballs also follow a standard.
- Top row: `Δbaseline` against `Δrelative` (the dissertation measure).
- Bottom row: `Δbaseline` against the change in **absolute** response, where the two axes are measured separately.

If the real value (vertical line) sits inside the grey histogram, the result is what chance gives.

In [ ]:
N_DRAWS = 1000
RANDOM_SEED = 0


def _masked(m, col):
    """Trials x ROIs values of the relative-analysis trials (NaN where the trial is not usable)."""
    return np.where(m["valid_rel"], m[col], np.nan)


def _nanmean_rows(a):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(a, axis=0)


def fake_oddball_null(block, condition, n_draws=N_DRAWS, seed=RANDOM_SEED):
    B_std, _ = roi_means(block, "standard", "base")
    R_std, _ = roi_means(block, "standard", "rel")
    A_std, _ = roi_means(block, "standard", "resp_rel")
    B_c, n_c = roi_means(block, condition, "base")
    R_c, _ = roi_means(block, condition, "rel")
    A_c, _ = roi_means(block, condition, "resp_rel")
    dB_real = B_c - B_std

    std = condition_matrices(block, "standard")
    pool = std["in_block"] & (std["prev_role"] == "standard") & std["valid_rel"].any(axis=1)
    base, rel, resp = (_masked(std, k)[pool] for k in ("base", "rel", "resp"))
    n_pick = int(np.median(n_c))

    rng = np.random.default_rng(seed)
    null = {k: np.full(n_draws, np.nan) for k in ("r_rel", "r_abs", "dB_mean", "dB_sd")}
    for k in range(n_draws):
        idx = rng.choice(base.shape[0], size=n_pick, replace=False)
        dB = _nanmean_rows(base[idx]) - B_std
        null["r_rel"][k] = pearson(dB, _nanmean_rows(rel[idx]) - R_std)
        null["r_abs"][k] = pearson(dB, _nanmean_rows(resp[idx]) - A_std)
        null["dB_mean"][k] = np.nanmean(dB)
        null["dB_sd"][k] = np.nanstd(dB)

    real = dict(r_rel=pearson(dB_real, R_c - R_std), r_abs=pearson(dB_real, A_c - A_std),
                dB_mean=float(np.nanmean(dB_real)), dB_sd=float(np.nanstd(dB_real)))

    def two_sided_p(x, dist):
        dist = dist[np.isfinite(dist)]
        centre = np.median(dist)
        return float((np.sum(np.abs(dist - centre) >= abs(x - centre)) + 1) / (dist.size + 1))

    p = {k: two_sided_p(real[k], null[k]) for k in real}
    return dict(real=real, null=null, p=p, n_pick=n_pick, n_pool=int(pool.sum()))


def plot_fake_oddball_null(block):
    conds = mismatch_conditions(block)
    res = {c: fake_oddball_null(block, c) for c in conds}
    fig, axs = plt.subplots(2, 3, figsize=(15, 7.5), sharex=True)
    rows = []
    for j, c in enumerate(conds):
        for i, (key, name) in enumerate([("r_rel", "corr(Δbaseline, Δrelative)"),
                                         ("r_abs", "corr(Δbaseline, Δabsolute)")]):
            ax = axs[i, j]
            dist = res[c]["null"][key]
            ax.hist(dist[np.isfinite(dist)], bins=40, color="0.75")
            ax.axvline(res[c]["real"][key], color=cond_color(block, c), lw=2.5)
            lo, hi = np.nanpercentile(dist, [2.5, 97.5])
            ax.set_title(f"{cond_label(block, c)}\nreal r = {res[c]['real'][key]:.2f}, "
                         f"chance {np.nanmedian(dist):.2f} [{lo:.2f}, {hi:.2f}], p = {res[c]['p'][key]:.3f}",
                         fontsize=11)
            ax.set_xlim(-1, 1)
            if j == 0:
                ax.set_ylabel(f"{name}\nnumber of draws")
            clean_axes(ax)
        for key, name in [("r_rel", "corr(Δbaseline, Δrelative)"), ("r_abs", "corr(Δbaseline, Δabsolute)"),
                          ("dB_mean", "mean Δbaseline over ROIs"), ("dB_sd", "SD of Δbaseline over ROIs")]:
            dist = res[c]["null"][key]
            lo, hi = np.nanpercentile(dist, [2.5, 97.5])
            rows.append({"block": block, "condition": cond_label(block, c), "measure": name,
                         "real": res[c]["real"][key], "chance median": np.nanmedian(dist),
                         "chance 2.5%": lo, "chance 97.5%": hi, "p (two-sided)": res[c]["p"][key],
                         "trials per draw": res[c]["n_pick"]})
    for ax in axs[1]:
        ax.set_xlabel("correlation across ROIs")
    fig.suptitle(f"Block {block}: real value (line) against {N_DRAWS} fake-oddball draws (grey)", y=1.0)
    fig.tight_layout()
    save_fig(fig, f"A1_fake_oddball_null_b{block}")
    plt.show()
    table = pd.DataFrame(rows)
    display(table.round(4))
    return res, table


fake_null = {}
fake_null_tables = []
for _b in BLOCKS:
    fake_null[_b], _tbl = plot_fake_oddball_null(_b)
    fake_null_tables.append(_tbl)
fake_null_table = pd.concat(fake_null_tables, ignore_index=True)

### 19.2 Split-half check: remove the shared noise

The mismatch trials of each condition are split at random into two halves.

- **Same half** (grey): `Δbaseline` and `Δrelative` come from the same trials, as in the dissertation analysis.
- **Different halves** (colour): `Δbaseline` comes from one half and `Δrelative` from the other, so they share no trial noise.

If ROIs with a truly larger baseline shift have a truly smaller relative response, the correlation stays negative across halves. If the effect is noise, it moves to about zero.

In [ ]:
N_SPLITS = 1000


def split_half_correlation(block, condition, n_splits=N_SPLITS, seed=RANDOM_SEED):
    B_std, _ = roi_means(block, "standard", "base")
    R_std, _ = roi_means(block, "standard", "rel")
    m = condition_matrices(block, condition)
    rows = m["in_block"] & m["valid_rel"].any(axis=1)
    base, rel = _masked(m, "base")[rows], _masked(m, "rel")[rows]
    n = base.shape[0]
    rng = np.random.default_rng(seed)
    r_same, r_cross = np.full(n_splits, np.nan), np.full(n_splits, np.nan)
    for k in range(n_splits):
        perm = rng.permutation(n)
        h1, h2 = perm[: n // 2], perm[n // 2:]
        dB_1 = _nanmean_rows(base[h1]) - B_std
        r_same[k]  = pearson(dB_1, _nanmean_rows(rel[h1]) - R_std)
        r_cross[k] = pearson(dB_1, _nanmean_rows(rel[h2]) - R_std)
    return dict(r_same=r_same, r_cross=r_cross, n_trials=int(n))


def plot_split_half(block):
    conds = mismatch_conditions(block)
    res = {c: split_half_correlation(block, c) for c in conds}
    fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
    rows = []
    bins = np.linspace(-1, 1, 41)
    for ax, c in zip(axs, conds):
        ax.hist(res[c]["r_same"], bins=bins, color="0.7", alpha=0.8, label="same half")
        ax.hist(res[c]["r_cross"], bins=bins, color=cond_color(block, c), alpha=0.6, label="different halves")
        ax.axvline(0, color="k", ls="--", lw=1)
        ax.set_title(f"{cond_label(block, c)} ({res[c]['n_trials']} trials)")
        ax.set_xlabel("corr(Δbaseline, Δrelative) across ROIs")
        clean_axes(ax)
        for key, name in [("r_same", "same half"), ("r_cross", "different halves")]:
            d = res[c][key]
            lo, hi = np.nanpercentile(d, [2.5, 97.5])
            rows.append({"block": block, "condition": cond_label(block, c), "halves": name,
                         "median r": np.nanmedian(d), "2.5%": lo, "97.5%": hi,
                         "fraction of splits with r < 0": float(np.nanmean(d < 0))})
    axs[0].set_ylabel("number of splits")
    axs[0].legend(frameon=False)
    fig.suptitle(f"Block {block}: split-half correlation ({N_SPLITS} random splits)", y=1.02)
    fig.tight_layout()
    save_fig(fig, f"A1_split_half_b{block}")
    plt.show()
    table = pd.DataFrame(rows)
    display(table.round(3))
    return res, table


split_half = {}
split_half_tables = []
for _b in BLOCKS:
    split_half[_b], _tbl = plot_split_half(_b)
    split_half_tables.append(_tbl)
split_half_table = pd.concat(split_half_tables, ignore_index=True)

### 19.3 Absolute baseline against absolute response, trial by trial

For each ROI, the absolute response of every trial is regressed on that trial's baseline. Both are measured separately, so neither is built from the other.

- **Slope near 1:** the response rides on top of the baseline. Subtracting the baseline is then a fair correction.
- **Slope below 1:** only part of the baseline carries into the response window. Subtracting the full baseline then over-corrects, and high-baseline trials get smaller relative responses.
- **Omission trials** have no stimulus, so their slope shows how much two neighbouring windows of the same trace are related anyway.

In [ ]:
def trial_level_slopes(block, min_trials=10):
    rows = []
    for c in ["standard"] + mismatch_conditions(block):
        m = condition_matrices(block, c)
        for r in range(N_ROI):
            ok = m["valid_rel"][:, r] & m["in_block"]
            x, y = m["base"][ok, r], m["resp"][ok, r]
            a, b, rr, n = linfit(x, y) if x.size >= min_trials else (np.nan, np.nan, np.nan, x.size)
            rows.append({"block": block, "condition": c, "roi": r, "n_trials": int(x.size),
                         "slope": a, "intercept": b, "r": rr})
    return pd.DataFrame(rows)


def plot_baseline_vs_response(block, roi=EXAMPLE_ROI):
    slopes = trial_level_slopes(block)
    conds = ["standard"] + mismatch_conditions(block)
    fig, axs = plt.subplots(1, 2, figsize=(13, 5))

    # left: one ROI, all standard trials
    m = condition_matrices(block, "standard")
    ok = m["valid_rel"][:, roi] & m["in_block"]
    x, y = m["base"][ok, roi], m["resp"][ok, roi]
    a, b, r, n = linfit(x, y)
    ax = axs[0]
    ax.scatter(x, y, s=6, color="0.4", alpha=0.4)
    lim = (float(min(x.min(), y.min())), float(max(x.max(), y.max())))
    ax.plot(lim, lim, "k--", lw=1, label="y = x")
    xx = np.linspace(lim[0], lim[1], 100)
    ax.plot(xx, a * xx + b, color="tab:green", lw=2, label=f"fit: y = {a:.2f}x + {b:.3f}, r = {r:.2f}")
    ax.set_xlabel("Baseline of the trial (absolute dF/F)")
    ax.set_ylabel("Response of the trial (absolute dF/F)")
    ax.set_title(f"Block {block}, ROI {roi}: {n} standard trials")
    ax.legend(frameon=False)
    clean_axes(ax)

    # right: slope of every ROI, per condition
    ax = axs[1]
    rng = np.random.default_rng(RANDOM_SEED)
    for i, c in enumerate(conds):
        s = slopes.loc[slopes["condition"] == c, "slope"].to_numpy()
        s = s[np.isfinite(s)]
        ax.scatter(i + rng.uniform(-0.12, 0.12, s.size), s, color=cond_color(block, c), alpha=0.8, s=30)
        if s.size >= 2:
            ax.errorbar(i + 0.28, s.mean(), yerr=s.std(ddof=1) / np.sqrt(s.size), fmt="o", color="k", capsize=4)
    ax.axhline(1, color="0.5", ls="--", lw=1)
    ax.axhline(0, color="0.5", ls=":", lw=1)
    ax.set_xticks(range(len(conds)))
    ax.set_xticklabels([cond_label(block, c) for c in conds])
    ax.set_ylabel("Slope of response on baseline (one dot per ROI)")
    ax.set_title(f"Block {block}: trial-level slopes (black = mean ± SEM over ROIs)")
    clean_axes(ax)
    fig.tight_layout()
    save_fig(fig, f"A1_baseline_vs_response_b{block}")
    plt.show()

    summary = (slopes.groupby("condition")
               .agg(rois=("slope", "count"), trials_per_roi=("n_trials", "median"),
                    mean_slope=("slope", "mean"), sd_slope=("slope", "std"), mean_r=("r", "mean"))
               .reindex(conds))
    summary.index = [cond_label(block, c) for c in conds]
    display(summary.round(3))
    return slopes


trial_slopes = pd.concat([plot_baseline_vs_response(b) for b in BLOCKS], ignore_index=True)

### 19.4 Global baseline: is the pre-stimulus state shared across the imaging plane?

`global_base` is the mean baseline of all other ROIs on the same trial (section 11). If baseline fluctuations reflect a shared state such as arousal, an ROI's own baseline should follow the global baseline, and the global baseline should predict the ROI's response.

For every ROI, on standard trials:

- **own vs global baseline:** how strongly the ROI's baseline follows the others (correlation across trials)
- **response on own baseline** and **response on global baseline:** slope and correlation across trials
- **both together:** response regressed on own and global baseline at once, to see which one carries the relation

In [ ]:
def global_baseline_analysis(block):
    sub = trial_table[(trial_table["block"] == block) & (trial_table["condition"] == "standard")
                      & trial_table["valid_rel"] & trial_table["in_block"] & trial_table["global_base"].notna()]
    rows = []
    for r, g in sub.groupby("roi"):
        own, glob, resp = g["base"].to_numpy(), g["global_base"].to_numpy(), g["resp"].to_numpy()
        a_own, _, r_own, _ = linfit(own, resp)
        a_glob, _, r_glob, _ = linfit(glob, resp)
        X = np.column_stack([own, glob, np.ones(own.size)])
        beta = np.linalg.lstsq(X, resp, rcond=None)[0]
        rows.append({"block": block, "roi": r, "trials": own.size,
                     "r(own baseline, global baseline)": pearson(own, glob),
                     "slope: response on own baseline": a_own, "r: response, own baseline": r_own,
                     "slope: response on global baseline": a_glob, "r: response, global baseline": r_glob,
                     "both: own baseline": beta[0], "both: global baseline": beta[1]})
    return pd.DataFrame(rows)


global_table = pd.concat([global_baseline_analysis(b) for b in BLOCKS], ignore_index=True)

_cols = ["r(own baseline, global baseline)", "r: response, own baseline", "r: response, global baseline"]
fig, axs = plt.subplots(1, len(BLOCKS), figsize=(6.5 * len(BLOCKS), 4.5), squeeze=False)
for ax, b in zip(axs[0], BLOCKS):
    sub = global_table[global_table["block"] == b]
    rng = np.random.default_rng(RANDOM_SEED)
    for j, col in enumerate(_cols):
        v = sub[col].to_numpy()
        ax.scatter(j + rng.uniform(-0.1, 0.1, v.size), v, color="0.4", s=28)
        ax.errorbar(j + 0.27, np.nanmean(v), yerr=sem(v), fmt="o", color="k", capsize=4)
    ax.axhline(0, color="0.5", ls="--", lw=1)
    ax.set_xticks(range(3))
    ax.set_xticklabels(["own baseline\nvs global baseline", "response\nvs own baseline", "response\nvs global baseline"])
    ax.set_ylabel("Correlation across standard trials (one dot per ROI)")
    ax.set_title(f"Block {b}")
    clean_axes(ax)
fig.tight_layout()
save_fig(fig, "global_baseline")
plt.show()
display(global_table.groupby("block").mean(numeric_only=True).drop(columns=["roi"]).round(3))

## 20. Across-block analysis (Block 1 vs Block 2)

The standard changes orientation between the blocks, so each orientation changes role or keeps it.

### 20.1 Standard baseline in Block 1 vs Block 2 (Supplementary Fig. 2)

In [ ]:
def plot_standard_baseline_across_blocks():
    B1, n1 = roi_means(1, "standard", "base")
    B2, n2 = roi_means(2, "standard", "base")
    m = np.isfinite(B1) & np.isfinite(B2)
    x, y = B1[m], B2[m]
    d = y - x
    a, b, r, n = linfit(x, y)
    l1, l2 = cond_label(1, "standard"), cond_label(2, "standard")

    print(f"=== Standard baseline: Block 1 ({l1}) vs Block 2 ({l2}) ===")
    print(f"Valid paired ROIs: {n} / {N_ROI}")
    print(f"Block 1 standard baseline mean ± SD: {x.mean():.4f} ± {x.std():.4f}")
    print(f"Block 2 standard baseline mean ± SD: {y.mean():.4f} ± {y.std():.4f}")
    print(f"Δ (Block 2 − Block 1) mean ± SD: {d.mean():+.4f} ± {d.std():.4f}")
    print(f"% ROIs with Block 2 baseline > Block 1 baseline: {(d > 0).mean() * 100:.1f}%")
    print(f"Fit: slope = {a:.4f}, intercept = {b:.4f}, r = {r:.4f}")
    print(f"Kept standard trials per ROI (median): Block 1 = {int(np.median(n1))}, Block 2 = {int(np.median(n2))}")

    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    lo, hi = float(min(x.min(), y.min())), float(max(x.max(), y.max()))
    margin = 0.05 * (hi - lo)
    lim = (lo - margin, hi + margin)
    axs[0].scatter(x, y, alpha=0.9)
    axs[0].plot(lim, lim, "k--", label="y = x")
    xx = np.linspace(lim[0], lim[1], 200)
    axs[0].plot(xx, a * xx + b, lw=2, label=f"fit: y = {a:.2f}x + {b:.3f}")
    axs[0].set_xlim(lim); axs[0].set_ylim(lim)
    axs[0].set_aspect("equal", adjustable="box")
    axs[0].set_xlabel(f"Block 1 standard baseline ({l1})")
    axs[0].set_ylabel(f"Block 2 standard baseline ({l2})")
    axs[0].legend(frameon=False)
    axs[1].scatter(x, d, alpha=0.9)
    axs[1].axhline(0, color="0.5", ls="--", lw=1)
    axs[1].set_xlabel("Block 1 standard baseline")
    axs[1].set_ylabel("Δbaseline (Block 2 − Block 1)")
    for ax in axs:
        clean_axes(ax)
    fig.tight_layout()
    save_fig(fig, "across_blocks_standard_baseline")
    plt.show()
    return dict(slope=a, intercept=b, r=r, n_roi=n, delta_mean=float(d.mean()), delta_sd=float(d.std()))


across_block_baseline = plot_standard_baseline_across_blocks()

### 20.2 Adaptation of the standard response within each block

For each ROI, the relative response to the standard early in the block (first 10% of kept trials) is compared with late in the block (last 10%), and a line is fitted across all trials.

- `AI_diff` = late mean − early mean
- `slope` = change in relative response per trial

In [ ]:
EARLY_FRAC = 0.10
LATE_FRAC  = 0.10
MIN_TRIALS_ADAPT = 200


def adaptation_metrics(block, early_frac=EARLY_FRAC, late_frac=LATE_FRAC, min_trials=MIN_TRIALS_ADAPT, eps=1e-6):
    m = condition_matrices(block, "standard")
    out = {k: np.full(N_ROI, np.nan) for k in ["n", "mu_early", "mu_late", "AI_ratio", "AI_diff", "slope"]}
    for r in range(N_ROI):
        rel = m["rel"][m["valid_rel"][:, r], r]          # time-ordered kept trials
        rel = rel[np.isfinite(rel)]
        n = rel.size
        out["n"][r] = n
        if n < min_trials:
            continue
        k1 = max(1, int(np.floor(n * early_frac)))
        k2 = max(1, int(np.floor(n * late_frac)))
        mu_early, mu_late = float(np.mean(rel[:k1])), float(np.mean(rel[-k2:]))
        slope, _ = np.polyfit(np.arange(n, dtype=float), rel, 1)
        out["mu_early"][r], out["mu_late"][r] = mu_early, mu_late
        out["AI_ratio"][r] = (mu_late - mu_early) / (mu_early + eps)
        out["AI_diff"][r] = mu_late - mu_early
        out["slope"][r] = float(slope)
    out["n"] = out["n"].astype(int)
    return out


adaptation = {b: adaptation_metrics(b) for b in BLOCKS}

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, key, name in [(axs[0], "AI_diff", "AI_diff (late − early)"), (axs[1], "slope", "slope (per trial)")]:
    x, y = adaptation[1][key], adaptation[2][key]
    a, b, r, n = linfit(x, y)
    m = np.isfinite(x) & np.isfinite(y)
    ax.scatter(x[m], y[m], alpha=0.9)
    if n >= 2:
        lim = (float(min(x[m].min(), y[m].min())), float(max(x[m].max(), y[m].max())))
        ax.plot(lim, lim, "k--", lw=1)
        xx = np.linspace(lim[0], lim[1], 200)
        ax.plot(xx, a * xx + b, lw=2, label=f"fit: y = {a:.2f}x + {b:.3g}, r = {r:.3f}")
        ax.set_aspect("equal", adjustable="box")
        ax.legend(frameon=False)
    ax.set_xlabel(f"Block 1 {name}")
    ax.set_ylabel(f"Block 2 {name}")
    ax.set_title(f"Standard adaptation: {key}")
    clean_axes(ax)
fig.tight_layout()
save_fig(fig, "across_blocks_adaptation")
plt.show()

print("Median across ROIs:")
for _b in BLOCKS:
    print(f"  Block {_b}: AI_diff = {np.nanmedian(adaptation[_b]['AI_diff']):+.6f}, "
          f"slope = {np.nanmedian(adaptation[_b]['slope']):+.6e}")

### 20.3 Response before and after the block switch (Fig. 8)

Four comparisons, each with Block 1 on the x-axis and Block 2 on the y-axis:

1. standard vs standard (different orientation, same role)
2. Block 1 standard vs the same orientation as an oddball in Block 2 (role switch)
3. 45° oddball vs 45° oddball (same orientation, same role)
4. Block 1 oddball vs the same orientation as the standard in Block 2 (role switch)

In [ ]:
def plot_pre_post(measure):
    s1, s2 = BLOCKS[1]["standard_ori"], BLOCKS[2]["standard_ori"]
    comparisons = [
        (f"Pre std {s1}° vs Post std {s2}°", (1, "standard"),        (2, "standard"),        "0.45"),
        (f"Pre std {s1}° vs Post odd {s1}°", (1, "standard"),        (2, f"oddball_{s1}"),   "tab:blue"),
        ("Pre odd 45° vs Post odd 45°",      (1, "oddball_45"),      (2, "oddball_45"),      "tab:purple"),
        (f"Pre odd {s2}° vs Post std {s2}°", (1, f"oddball_{s2}"),   (2, "standard"),        "tab:red"),
    ]
    pairs = [(name, roi_means(*pre, measure)[0], roi_means(*post, measure)[0], col)
             for name, pre, post, col in comparisons]

    vals = np.concatenate([np.r_[x[np.isfinite(x) & np.isfinite(y)], y[np.isfinite(x) & np.isfinite(y)]]
                           for _, x, y, _ in pairs])
    lo, hi = float(vals.min()), float(vals.max())
    pad = 0.06 * (hi - lo + 1e-12)
    lim = (lo - pad, hi + pad)

    unit = "mean dF/F" if measure == "abs" else "relative dF/F"
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    results = {}
    for ax, (name, x, y, col) in zip(axs, pairs):
        a, b, r, n = linfit(x, y)
        m = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[m], y[m], s=26, alpha=0.9, color=col, edgecolor="none")
        ax.plot(lim, lim, "k--", lw=1, alpha=0.8)
        if n >= 2:
            grid = np.linspace(lim[0], lim[1], 100)
            ax.plot(grid, a * grid + b, color=col, lw=2)
        ax.text(0.03, 0.97, f"n={n}\nfit: y={a:.2f}x{b:+.3f}\nr={r:.2f}", transform=ax.transAxes,
                ha="left", va="top", fontsize=13)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(name.split(" vs ")[0])
        ax.set_ylabel(name.split(" vs ")[1])
        ax.set_title(name)
        clean_axes(ax)
        results[name] = dict(slope=a, intercept=b, r=r, n_roi=n)
    fig.suptitle(f"Pre vs post comparisons ({unit})", fontsize=14, y=1.03)
    fig.tight_layout()
    save_fig(fig, f"across_blocks_pre_post_{measure}")
    plt.show()
    display(pd.DataFrame(results).T.round(4))
    return results


pre_post = {m: plot_pre_post(m) for m in ["abs", "rel"]}

### 20.4 Slope checks for the before/after fits

The same checks as in section 16.4. A before/after slope is pulled towards zero whenever the x-axis is a mismatch condition with about 30 trials, because those per-ROI values are noisy. The reliability column shows how much, and the corrected slope undoes it where the reliability is high enough.

In [ ]:
_s1, _s2 = BLOCKS[1]["standard_ori"], BLOCKS[2]["standard_ori"]
_pairs = [
    (f"Pre std {_s1}° vs Post std {_s2}°", "standard",        "standard"),
    (f"Pre std {_s1}° vs Post odd {_s1}°", "standard",        f"oddball_{_s1}"),
    ("Pre odd 45° vs Post odd 45°",        "oddball_45",      "oddball_45"),
    (f"Pre odd {_s2}° vs Post std {_s2}°", f"oddball_{_s2}",  "standard"),
]
pre_post_check_table = pd.DataFrame([slope_check_row(name, 1, cx, 2, cy, m)
                                     for m in ["abs", "rel"] for name, cx, cy in _pairs])
display(pre_post_check_table.round(3))

## 21. Adaptation over repetitions of the standard

Two views of how the response to the standard changes as it is repeated:

- **Since the last mismatch trial:** the response as a function of how many standards came in a row before it. `0` is the first standard after an oddball or omission.
- **Over the block:** the response in 20 successive parts of the block.

Both are shown for the absolute and the relative response, as the mean ± SEM over ROIs. The green line is the same grating in the control trials (not repeated).

In [ ]:
REP_BINS = [(0, 0), (1, 1), (2, 2), (3, 4), (5, 9), (10, 19), (20, 10**6)]
REP_LABELS = ["0", "1", "2", "3–4", "5–9", "10–19", "20+"]


def repetition_curves(block, measure):
    m = condition_matrices(block, "standard")
    col, valid = MEASURES[measure]
    vals = np.where(m[valid] & m["in_block"][:, None], m[col], np.nan)
    reps = m["n_std_before"]
    by_rep = np.array([nanmean_quiet(vals[(reps >= lo) & (reps <= hi)], axis=0) for lo, hi in REP_BINS])   # bins x ROIs
    n_rep = [int(((reps >= lo) & (reps <= hi)).sum()) for lo, hi in REP_BINS]
    parts = np.array_split(np.arange(vals.shape[0]), 20)
    by_time = np.array([nanmean_quiet(vals[p], axis=0) for p in parts])                                     # 20 x ROIs
    return by_rep, n_rep, by_time


fig, axs = plt.subplots(2, 2 * len(BLOCKS), figsize=(5.2 * 2 * len(BLOCKS), 8), squeeze=False)
_rows = []
for j, (b, measure) in enumerate([(b, m) for b in BLOCKS for m in ["abs", "rel"]]):
    by_rep, n_rep, by_time = repetition_curves(b, measure)
    control_level = float(np.nanmean(control_means(BLOCKS[b]["standard_ori"], measure)[0]))
    name = "absolute" if measure == "abs" else "relative"

    ax = axs[0, j]
    ax.errorbar(range(len(REP_BINS)), np.nanmean(by_rep, axis=1), yerr=sem(by_rep, axis=1), color="k", marker="o", capsize=3)
    ax.axhline(control_level, color="tab:green", lw=1.5, label="same grating, control trials")
    ax.set_xticks(range(len(REP_BINS)))
    ax.set_xticklabels(REP_LABELS)
    ax.set_xlabel("Standards in a row before this trial")
    ax.set_ylabel(f"{name.capitalize()} response")
    ax.set_title(f"Block {b}, {cond_label(b, 'standard')}, {name}", fontsize=12)
    ax.legend(frameon=False, fontsize=9)
    clean_axes(ax)

    ax = axs[1, j]
    ax.errorbar(np.arange(1, 21), np.nanmean(by_time, axis=1), yerr=sem(by_time, axis=1), color="k", marker="o", ms=3, capsize=2)
    ax.axhline(control_level, color="tab:green", lw=1.5)
    ax.set_xlabel("Part of the block (1 = start, 20 = end)")
    ax.set_ylabel(f"{name.capitalize()} response")
    clean_axes(ax)

    first, later = by_rep[0], by_rep[-1]
    slope_time = np.array([np.polyfit(np.arange(20), by_time[:, r], 1)[0] if np.isfinite(by_time[:, r]).all() else np.nan
                           for r in range(N_ROI)])
    _rows.append({"block": b, "measure": measure, "control level": control_level,
                  "first standard after a mismatch (0)": np.nanmean(first), "after 20+ standards": np.nanmean(later),
                  "difference (20+ minus 0)": np.nanmean(later - first), "p (paired, ROIs)": paired_p(later, first),
                  "trials in bin 0": n_rep[0], "trials in bin 20+": n_rep[-1],
                  "change per part of the block": np.nanmean(slope_time), "p of that change (ROIs)": paired_p(slope_time)})
fig.tight_layout()
save_fig(fig, "adaptation_over_repetitions")
plt.show()
repetition_table = pd.DataFrame(_rows)
display(repetition_table.round(4))

## 22. Session summary and export

All fitted values of this session in one long table, saved as a CSV file. The trial-level tables are saved too, so the sessions can be combined later for the across-mouse analysis.

In [ ]:
_rows = []
def _add(analysis, block, measure, condition, metric, value):
    _rows.append(dict(mouse=MOUSE_ID, dmd=DMD, analysis=analysis, block=block, measure=measure,
                      condition=condition, metric=metric, value=value))

# --- standard vs oddball fits ---
for (b, measure, trim), res in wedge_results.items():
    for c, f in res["fits"].items():
        for metric in ("slope", "intercept", "r"):
            _add(f"standard_vs_oddball (trim={trim})", b, measure, c, metric, f[metric])
    _add(f"standard_vs_oddball (trim={trim})", b, measure, "both", "n_roi_used", res["n_roi_used"])
    _add(f"standard_vs_oddball (trim={trim})", b, measure, "both", "area_signed", res["area_signed"])
for (b, measure), res in standard_vs_mismatch.items():
    for metric in ("slope", "intercept", "r"):
        _add("standard_vs_omission", b, measure, "omission", metric, res["omission"][metric])
for _, row in slope_check_table.iterrows():
    for metric in ("slope", "slope 2.5%", "slope 97.5%", "through-origin slope", "Deming slope",
                   "reliability of x", "reliability of y", "corrected slope"):
        _add("slope_checks", row["comparison"][:2], row["measure"], row["comparison"], metric, row[metric])

# --- pooled over ROIs, normalised, role switch, context ---
for _, row in population_table.iterrows():
    for metric in ("effect (mean over ROIs)", "effect in chance SDs", "p"):
        _add("population_test", row["block"], row["measure"], row["condition"], metric, row[metric])
for _, row in normalised_summary.iterrows():
    for metric in ("mean z", "ROIs with |z| > 1.96", "mean modulation index", "level vs standard r"):
        _add("normalised_effect", row["block"], "abs", row["condition"], metric, row[metric])
for _, row in role_table.iterrows():
    for metric in ("mean over ROIs", "SEM over ROIs", "p (paired, ROIs)"):
        _add("role_switch", "1 vs 2", row["measure"], row["contrast"], metric, row[metric])
for _, row in context_table.iterrows():
    for metric in ("mean over ROIs", "difference from control", "p vs control (paired, ROIs)"):
        _add("context", "all", row["measure"], f"{row['grating']}° {row['context']}", metric, row[metric])
for b, res in tuning_vs_oddball.items():
    for metric in ("slope", "r"):
        _add("tuning_vs_oddball", b, "rel", "oddball difference", metric, res[metric])
_add("tuning", "orientation blocks", "rel", "all", "ROIs tuned at p<0.05", int((tuning_table["tuning test p"] < 0.05).sum()))

# --- around the oddball, repetitions ---
for _, row in around_table.iterrows():
    for metric in ("mean", "p"):
        _add("around_oddball", row["block"], "abs", f"{row['condition']}: {row['difference']}", metric, row[metric])
for _, row in repetition_table.iterrows():
    for metric in ("difference (20+ minus 0)", "p (paired, ROIs)", "change per part of the block", "control level"):
        _add("adaptation_over_repetitions", row["block"], row["measure"], "standard", metric, row[metric])

# --- baseline ---
for b, res in baseline_scatter.items():
    for c, f in res.items():
        for metric in ("slope", "intercept", "r", "delta_mean", "delta_sd"):
            _add("baseline_scatter", b, "base", c, metric, f[metric])
for b, res in baseline_results.items():
    for c, f in res.items():
        for metric in ("corr_dB_dRel", "state_corr", "state_slope", "n_roi"):
            _add("baseline_vs_relative", b, "rel", c, metric, f[metric])
for b, res in fake_null.items():
    for c, f in res.items():
        for key in ("r_rel", "r_abs", "dB_mean", "dB_sd"):
            _add("A1_fake_oddball_null", b, key, c, "real", f["real"][key])
            _add("A1_fake_oddball_null", b, key, c, "chance_median", float(np.nanmedian(f["null"][key])))
            _add("A1_fake_oddball_null", b, key, c, "p_two_sided", f["p"][key])
for b, res in split_half.items():
    for c, f in res.items():
        _add("A1_split_half", b, "rel", c, "median_r_same_half", float(np.nanmedian(f["r_same"])))
        _add("A1_split_half", b, "rel", c, "median_r_different_halves", float(np.nanmedian(f["r_cross"])))
for (b, c), g in trial_slopes.groupby(["block", "condition"]):
    _add("A1_trial_level_slope", b, "abs", c, "mean_slope", float(g["slope"].mean()))
for b, g in global_table.groupby("block"):
    for metric in ("r(own baseline, global baseline)", "r: response, own baseline", "r: response, global baseline",
                   "both: own baseline", "both: global baseline"):
        _add("global_baseline", b, "abs", "standard", metric, float(g[metric].mean()))

# --- across blocks ---
for metric in ("slope", "intercept", "r", "delta_mean", "delta_sd"):
    _add("across_blocks_standard_baseline", "1 vs 2", "base", "standard", metric, across_block_baseline[metric])
for measure, res in pre_post.items():
    for name, f in res.items():
        for metric in ("slope", "intercept", "r"):
            _add("across_blocks_pre_post", "1 vs 2", measure, name, metric, f[metric])
for _, row in pre_post_check_table.iterrows():
    for metric in ("reliability of x", "reliability of y", "corrected slope", "Deming slope"):
        _add("across_blocks_pre_post_checks", "1 vs 2", row["measure"], row["comparison"], metric, row[metric])

session_summary = pd.DataFrame(_rows)
session_summary.to_csv(f"{MOUSE_ID}_{DMD}_summary.csv", index=False)
trial_table.to_csv(f"{MOUSE_ID}_{DMD}_trial_table.csv", index=False)
control_table.to_csv(f"{MOUSE_ID}_{DMD}_control_table.csv", index=False)
print(f"Saved {len(session_summary)} summary rows to {MOUSE_ID}_{DMD}_summary.csv")
print(f"Saved the trial-level tables to {MOUSE_ID}_{DMD}_trial_table.csv and {MOUSE_ID}_{DMD}_control_table.csv")
display(session_summary.head(12))